In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('ggplot')
import duckdb
import pyarrow

In [ ]:
# data= pd.read_csv("/gpfs/user_home/os_home_dirs/zsherif2/spring23_equifax/spring23_equifax.csv",low_memory=True )

In [ ]:
import duckdb

In [2]:
file_path = "/gpfs/user_home/os_home_dirs/zsherif2/spring23_equifax/spring23_equifax.csv"
desired_revisions = [201807, 201907, 202007]

In [ ]:
# con = duckdb.connect(database=':memory:')

In [ ]:
# try:
#     # Use DuckDB's SQL interface to read the CSV and filter it
#     # DuckDB's `read_csv_auto` (or just using the file path in FROM)
#     # is highly optimized for performance and will handle type inference
#     # and out-of-core processing.
#     # The WHERE clause ensures that only the filtered rows are processed and returned.
#     query = f"""
#     SELECT *
#     FROM read_csv_auto('{file_path}')
#     WHERE META_REVISION IN ({', '.join(map(str, desired_revisions))});
#     """

#     # Execute the query and fetch the results directly into a Pandas DataFrame
#     filtered_df = con.execute(query).fetchdf()

#     print(f"Filtered dataset loaded with DuckDB. Number of rows: {len(filtered_df)}")

#     # Display the first few rows of the filtered DataFrame
#     print("\nFirst 5 rows of the filtered dataset:")
#     print(filtered_df.head())

#     # Verify the values in 'META_REVISION' column
#     print("\nUnique values in 'META_REVISION' of the filtered dataset:")
#     print(filtered_df['META_REVISION'].unique())

# except duckdb.Error as e:
#     print(f"DuckDB Error: {e}")
# except FileNotFoundError:
#     print(f"Error: The file '{file_path}' was not found. Please check the path.")
# except Exception as e:
#     print(f"An unexpected error occurred: {e}")
# finally:
#     # Close the connection
#     con.close()

In [3]:

import pyarrow

def process_large_dataset(input_file_path: str, output_file_path: str = None) -> pd.DataFrame:
    """
    Cleans and filters a large dataset and optionally saves the result to a file.

    Args:
        input_file_path: The path to the large dataset (e.g., 'your_data.parquet').
        output_file_path: Optional. The path to save the cleaned output file to
                          (e.g., 'cleaned_data.parquet' or 'cleaned_data.csv').

    Returns:
        A Pandas DataFrame containing the cleaned and filtered data.
    """
    print(f"🚀 Starting processing for {input_file_path}...")
    # Connect to an in-memory DuckDB database
    con = duckdb.connect(database=':memory:', read_only=False)

    # Create a view on the file to query it from disk
    try:
        con.execute(f"CREATE OR REPLACE VIEW raw_data AS SELECT * FROM read_csv_auto('{input_file_path}')")
        table_info = con.execute("PRAGMA table_info('raw_data')").fetchdf()
    except Exception as e:
        print(f"❌ Error reading input file: {e}")
        con.close()
        return pd.DataFrame()

    select_clauses = []
    
    # Dynamically build the SELECT clauses based on column types
    column_names = [name.upper() for name in table_info['name']]
    if 'UNIQUE_CONSUMER_KEY' in column_names:
        select_clauses.append('"UNIQUE_CONSUMER_KEY"')
    if 'META_REVISION' in column_names:
        select_clauses.append('"META_REVISION"')

    for _, row in table_info.iterrows():
        col_name = row['name']
        if col_name.upper() in ['UNIQUE_CONSUMER_KEY', 'META_REVISION']:
            continue
        
        col_type = row['type'].upper()
        
        if col_type in ['FLOAT', 'DOUBLE', 'REAL', 'DECIMAL']:
            clause = f"""
            CASE
                WHEN "{col_name}" BETWEEN 9.9993 AND 9.9999 THEN NULL
                WHEN "{col_name}" BETWEEN 99.9993 AND 99.9999 THEN NULL
                ELSE "{col_name}"
            END AS "{col_name}"
            """
            select_clauses.append(clause)
        elif 'INT' in col_type:
            clause = f"""
            CASE
                WHEN "{col_name}" BETWEEN 2 AND 9 THEN NULL
                WHEN "{col_name}" BETWEEN 93 AND 99 THEN NULL
                WHEN "{col_name}" BETWEEN 993 AND 999 THEN NULL
                WHEN "{col_name}" BETWEEN 999999993 AND 999999999 THEN NULL
                ELSE "{col_name}"
            END AS "{col_name}"
            """
            select_clauses.append(clause)
        else:
            select_clauses.append(f'"{col_name}"')

    # Assemble and execute the final query
    final_query = f"""
    SELECT
        {', '.join(select_clauses)}
    FROM
        raw_data
    WHERE
        META_REVISION IN (202007,201907, 201807)
    """
    result_df = con.execute(final_query).arrow().to_pandas()
    con.close()
    print(f"✅ Processing complete. {len(result_df)} rows generated.")

    # --- Save the output if a path is provided ---
    if output_file_path:
        print(f"💾 Saving output to {output_file_path}...")
        try:
            if output_file_path.lower().endswith('.parquet'):
                result_df.to_parquet(output_file_path)
                print("   File saved successfully as Parquet.")
            elif output_file_path.lower().endswith('.csv'):
                result_df.to_csv(output_file_path, index=False)
                print("   File saved successfully as CSV.")
            else:
                print(f"   ⚠️ Warning: Unknown file extension. Please use '.parquet' or '.csv'. File not saved.")
        except Exception as e:
            print(f"   ❌ Error saving file: {e}")

    return result_df

In [3]:
def process_large_dataset_todisk(input_file_path: str, output_file_path: str) -> None:
    """
    Cleans and filters a large dataset and saves the result directly to a file
    without loading into memory.
    
    Args:
        input_file_path: The path to the large dataset (e.g., 'your_data.parquet').
        output_file_path: The path to save the cleaned output file to
                          (e.g., 'cleaned_data.parquet' or 'cleaned_data.csv').
    """
    print(f"🚀 Starting processing for {input_file_path}...")
    
    # Connect to an in-memory DuckDB database
    con = duckdb.connect(database=':memory:', read_only=False)
    
    try:
        # Create a view on the file to query it from disk
        con.execute(f"CREATE OR REPLACE VIEW raw_data AS SELECT * FROM read_csv_auto('{input_file_path}')")
        table_info = con.execute("PRAGMA table_info('raw_data')").fetchdf()
        
        select_clauses = []
        
        # Dynamically build the SELECT clauses based on column types
        column_names = [name.upper() for name in table_info['name']]
        if 'UNIQUE_CONSUMER_KEY' in column_names:
            select_clauses.append('"UNIQUE_CONSUMER_KEY"')
        if 'META_REVISION' in column_names:
            select_clauses.append('"META_REVISION"')
            
        for _, row in table_info.iterrows():
            col_name = row['name']
            if col_name.upper() in ['UNIQUE_CONSUMER_KEY', 'META_REVISION']:
                continue
            
            col_type = row['type'].upper()
            
            if col_type in ['FLOAT', 'DOUBLE', 'REAL', 'DECIMAL']:
                clause = f"""
                CASE
                    WHEN "{col_name}" BETWEEN 9.9993 AND 9.9999 THEN NULL
                    WHEN "{col_name}" BETWEEN 99.9993 AND 99.9999 THEN NULL
                    ELSE "{col_name}"
                END AS "{col_name}"
                """
                select_clauses.append(clause)
            elif 'INT' in col_type:
                clause = f"""
                CASE
                    WHEN "{col_name}" BETWEEN 2 AND 9 THEN NULL
                    WHEN "{col_name}" BETWEEN 93 AND 99 THEN NULL
                    WHEN "{col_name}" BETWEEN 993 AND 999 THEN NULL
                    WHEN "{col_name}" BETWEEN 999999993 AND 999999999 THEN NULL
                    ELSE "{col_name}"
                END AS "{col_name}"
                """
                select_clauses.append(clause)
            else:
                select_clauses.append(f'"{col_name}"')
        
        # Assemble the transformation query
        transformation_query = f"""
        SELECT
            {', '.join(select_clauses)}
        FROM
            raw_data
        WHERE
            META_REVISION IN (202007, 201907, 201807)
        """
        
        # Determine output format and write directly to file
        print(f"💾 Processing and saving directly to {output_file_path}...")
        
        if output_file_path.lower().endswith('.parquet'):
            # Write directly to Parquet
            copy_query = f"""
            COPY ({transformation_query})
            TO '{output_file_path}'
            (FORMAT PARQUET)
            """
            con.execute(copy_query)
            print("   ✅ File saved successfully as Parquet.")
            
        elif output_file_path.lower().endswith('.csv'):
            # Write directly to CSV
            copy_query = f"""
            COPY ({transformation_query})
            TO '{output_file_path}'
            (FORMAT CSV, HEADER)
            """
            con.execute(copy_query)
            print("   ✅ File saved successfully as CSV.")
            
        else:
            print(f"   ⚠️ Warning: Unknown file extension. Please use '.parquet' or '.csv'.")
            return
        
        # Get row count for reporting (this is a lightweight operation)
        row_count = con.execute(f"SELECT COUNT(*) FROM ({transformation_query})").fetchone()[0]
        print(f"✅ Processing complete. {row_count} rows processed and saved.")
        
    except Exception as e:
        print(f"❌ Error processing file: {e}")
        
    finally:
        con.close()

In [4]:
output_path = '/gpfs/user_home/os_home_dirs/zsherif2/test/efx2018_2019_2020_cleaned.parquet'

In [7]:
%whos

Variable                Type        Data/Info
---------------------------------------------
desired_revisions       list        n=3
duckdb                  module      <module 'duckdb' from '/g<...>ages/duckdb/__init__.py'>
file_path               str         /gpfs/user_home/os_home_d<...>ifax/spring23_equifax.csv
np                      module      <module 'numpy' from '/gp<...>kages/numpy/__init__.py'>
output_path             str         /gpfs/user_home/os_home_d<...>2019_2020_cleaned.parquet
pd                      module      <module 'pandas' from '/g<...>ages/pandas/__init__.py'>
process_large_dataset   function    <function process_large_d<...>ataset at 0x7f275ed5e520>
pyarrow                 module      <module 'pyarrow' from '/<...>ges/pyarrow/__init__.py'>


In [8]:
import gc
gc.collect()

1436

In [ ]:
final_df = process_large_dataset(file_path, output_path)

In [5]:
process_large_dataset_todisk(file_path, output_path)

🚀 Starting processing for /gpfs/user_home/os_home_dirs/zsherif2/spring23_equifax/spring23_equifax.csv...
💾 Processing and saving directly to /gpfs/user_home/os_home_dirs/zsherif2/test/efx2018_2019_2020_cleaned.parquet...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   ✅ File saved successfully as Parquet.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Processing complete. 8781222 rows processed and saved.


In [2]:
input_parquet_path = '/gpfs/user_home/os_home_dirs/zsherif2/test/efx2018_2019_2020_cleaned.parquet'
output_parquet_path = '/gpfs/user_home/os_home_dirs/zsherif2/test/efx2018_2019_2020_target.parquet'
col_to_check = 'CRP02_0206015060'

In [3]:
con = duckdb.connect()

try:
    # --- 1. Check if the column exists in the Parquet file ---
    schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{input_parquet_path}')").fetchall()
    column_names = [col[0] for col in schema]

    if col_to_check in column_names:
        print(f"✅ Found column '{col_to_check}'. Processing...")
        
        # --- 2. Dynamically build the SELECT statement ---
        select_clauses = []
        print("Analyzing schema to cast float columns...")
        
        for col_name, col_type, _, _, _, _ in schema:
            # Skip the original column that we are dropping/transforming
            if col_name == col_to_check:
                continue

            # If the column is a 64-bit float (DOUBLE), cast it to 32-bit (FLOAT)
            if col_type == 'DOUBLE':
                select_clauses.append(f'CAST("{col_name}" AS FLOAT) AS "{col_name}"')
            else:
                # Keep all other columns as they are
                select_clauses.append(f'"{col_name}"')

        # Add the 'target' column transformation logic
        target_clause = f'(CASE WHEN COALESCE("{col_to_check}", 0) > 0 THEN 1 ELSE 0 END)::UTINYINT AS target'
        select_clauses.append(target_clause)

        # Join all parts into the final SELECT statement
        final_select_statement = ",\n            ".join(select_clauses)

        # --- 3. Execute the full query to transform and save ---
        query = f"""
        COPY (
            SELECT
                {final_select_statement}
            FROM read_parquet('{input_parquet_path}')
        ) TO '{output_parquet_path}' (FORMAT PARQUET);
        """
        
        print("\nExecuting transformation and saving the file...")
        con.execute(query)
        
        print(f"🚀 Processing complete. New file with float32 columns saved to:\n{output_parquet_path}")

    else:
        print(f"❌ Column '{col_to_check}' not found in the Parquet file.")

except duckdb.Error as e:
    print(f"An error occurred: {e}")
finally:
    # Close the database connection
    con.close()

✅ Found column 'CRP02_0206015060'. Processing...
Analyzing schema to cast float columns...

Executing transformation and saving the file...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

🚀 Processing complete. New file with float32 columns saved to:
/gpfs/user_home/os_home_dirs/zsherif2/test/efx2018_2019_2020_target.parquet


In [3]:


# Define the path to your Parquet file
file_path = '/gpfs/user_home/os_home_dirs/zsherif2/test/efx2018_2019_2020_target.parquet'

# List of revision values you want to load
revisions_to_load = [202007]

# A dictionary to hold the separate DataFrames
dataframes = {}

# Loop through each revision and load it separately
for revision in revisions_to_load:
    print(f"Loading data for META_REVISION == {revision}...")
    
    # The filter is a list of tuples: [(column, operator, value)]
    try:
        df = pd.read_parquet(
            file_path,
            filters=[('META_REVISION', '==', revision)]
        )
        
        # Store the DataFrame in the dictionary with a descriptive key
        dataframes[f'df_{revision}'] = df
        
        print(f"Successfully loaded {len(df)} rows for revision {revision}.")
        # print(df.head()) # Optional: uncomment to see the first few rows
        print("-" * 30)

    except Exception as e:
        print(f"Could not load data for revision {revision}. Error: {e}")

# Now you can access each DataFrame from the dictionary
# For example, to access the data for 201807:
# df_201807 = dataframes.get('df_201807')

# if df_201807 is not None:
#     print("\nAccessing the 201807 DataFrame:")
#     print(df_201807.head())

Loading data for META_REVISION == 202007...
Successfully loaded 2859875 rows for revision 202007.
------------------------------


In [5]:
df_201807.to_parquet('df_201807.parquet')

In [9]:
df_201907=dataframes.get('df_201907')
df_201907.to_parquet('df_201907.parquet')

In [4]:
df_202007=dataframes.get('df_202007')
df_202007.to_parquet('df_202007.parquet')

In [2]:


# --- Configuration ---
# ⚠️ Update these paths to your files
input_parquet_path = '/gpfs/user_home/os_home_dirs/zsherif2/test/df_201807.parquet'
output_parquet_path = '/gpfs/user_home/os_home_dirs/zsherif2/test/efx2018_no30%missing.parquet'
missing_value_threshold = 30.0  # Drop columns with more than 30% missing values

# Use an in-memory DuckDB database
con = duckdb.connect()

try:
    print(f"🚀 Analyzing file: {input_parquet_path}")

    # --- 1. Get column statistics in a single pass ---
    # The SUMMARIZE command is highly efficient for this task.
    # We convert the small summary result to a pandas DataFrame for easy filtering.
    summary_df = con.sql(f"""
        SUMMARIZE SELECT * FROM read_parquet('{input_parquet_path}')
    """).df()

    # --- 2. Identify columns to keep and drop ---
    # The 'null_percentage' column from SUMMARIZE is used here.
    cols_to_drop = summary_df[summary_df['null_percentage'] > missing_value_threshold]['column_name'].tolist()
    all_cols = summary_df['column_name'].tolist()
    cols_to_keep = [col for col in all_cols if col not in cols_to_drop]

    if not cols_to_drop:
        print("✅ No columns exceeded the 30% missing value threshold.")
    else:
        print(f"\nIdentified {len(cols_to_drop)} columns to drop:")
        for col in cols_to_drop:
            print(f"  - {col}")

        print(f"\nKeeping {len(cols_to_keep)} columns.")
        
        # --- 3. Build query and save the new dataset ---
        # We construct the SELECT statement with only the columns we want to keep.
        # The f'"{col}"' syntax correctly quotes column names.
        select_statement = ",\n        ".join([f'"{col}"' for col in cols_to_keep])

        query = f"""
        COPY (
            SELECT
                {select_statement}
            FROM read_parquet('{input_parquet_path}')
        ) TO '{output_parquet_path}' (FORMAT PARQUET);
        """
        
        print("\nSaving the final dataset without the dropped columns...")
        con.execute(query)
        print(f"✨ Success! Final cleaned file saved to:\n{output_parquet_path}")

except duckdb.Error as e:
    print(f"An error occurred: {e}")
finally:
    # Close the database connection
    con.close()

🚀 Analyzing file: /gpfs/user_home/os_home_dirs/zsherif2/test/df_201807.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Identified 347 columns to drop:
  - CRP02_0101080000
  - CRP02_0101350000
  - CRP02_0101410000
  - CRP02_0101430000
  - CRP02_0101440000
  - CRP02_0102010000
  - CRP02_0102080000
  - CRP02_0102090000
  - CRP02_0102120000
  - CRP02_0102190000
  - CRP02_0102290000
  - CRP02_0102320000
  - CRP02_0102330000
  - CRP02_0102350000
  - CRP02_0102440000
  - CRP02_0103090000
  - CRP02_0105090000
  - CRP02_0105110000
  - CRP02_0105120000
  - CRP02_0108090060
  - CRP02_0113015000
  - CRP02_0113105000
  - CRP02_0113125000
  - CRP02_0113185000
  - CRP02_0113295000
  - CRP02_0113325000
  - CRP02_0113355000
  - CRP02_0113475000
  - CRP02_0113585000
  - CRP02_0114575000
  - CRP02_0115080000
  - CRP02_0117090000
  - CRP02_0118090000
  - CRP02_0119090000
  - CRP02_0143090000
  - CRP02_0143120000
  - CRP02_0143290000
  - CRP02_0205107500
  - CRP02_0206015020
  - CRP02_0206015030
  - CRP02_0206015040
  - CRP02_0206125020
  - CRP02_0206125030
  - CRP02_0206125040
  - CRP02_0206125050
  - CRP02_0206125060
 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✨ Success! Final cleaned file saved to:
/gpfs/user_home/os_home_dirs/zsherif2/test/efx2018_no30%missing.parquet


In [5]:
#reading cleaned file from disk
final_df= pd.read_parquet('/gpfs/user_home/os_home_dirs/zsherif2/test/efx2018_no30%missing.parquet')

In [6]:
col_to_keep= final_df.columns.tolist()

In [7]:


# Use DuckDB to select columns and save
import duckdb
con = duckdb.connect()
cols_str = ', '.join([f'"{col}"' for col in col_to_keep])

con.execute(f"""
COPY (SELECT {cols_str} FROM 'df_202007.parquet')
TO 'efx2020_no30%missing.parquet' (FORMAT PARQUET)
""")
con.close()

# Then load the much smaller result
efx2020 = pd.read_parquet('efx2020_no30%missing.parquet')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [11]:
final_df.head()

,UNIQUE_CONSUMER_KEY,META_REVISION,VANTAGE40,CRP02_0101010000,CRP02_0101110000,CRP02_0101120000,CRP02_0101290000,CRP02_0101470000,CRP02_0105010000,CRP02_0105030000,...,CRP02_0908015020,CRP02_0908015030,CRP02_0908015040,CRP02_0908015060,CRP02_0908475020,CRP02_0908585020,CRP02_0910015000,CRP02_0910035000,CRP02_0913015010,target
0,Ykyr4dyJD464pImC1wQL2,201807,748.0,157.0,144.0,157.0,80.0,157.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.1852,0.1923,0.3571,0
1,OBoyc20SLCDfChNQWuTaT,201807,606.0,88.0,88.0,30.0,76.0,88.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.3030,0.3030,0.7083,0
2,fa8Qy4KsUmVQwHJd80Oxq,201807,569.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,pxmUMGn663v1hlM1Zy0YV,201807,559.0,33.0,NaN,NaN,NaN,33.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0000,1.0000,0
4,TmiP6ORfV4BTGIsVN1Li0,201807,759.0,178.0,127.0,127.0,178.0,133.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0000,0.2000,0


In [8]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000000 entries, 0 to 2999999
Columns: 141 entries, UNIQUE_CONSUMER_KEY to target
dtypes: float32(34), float64(104), int64(1), object(1), uint8(1)
memory usage: 2.8+ GB


In [3]:
df_2018=pd.read_parquet('/gpfs/user_home/os_home_dirs/zsherif2/test/efx2018_no30%missing.parquet')
df_2019=pd.read_parquet('/gpfs/user_home/os_home_dirs/zsherif2/test/efx2019_no30%missing.parquet')
df_2020=pd.read_parquet('/gpfs/user_home/os_home_dirs/zsherif2/test/efx2020_no30%missing.parquet')

In [17]:
def create_lagged_dataset(features_df, target_df, key_col='UNIQUE_CONSUMER_KEY'):
    """Create dataset with features from features_df and targets from target_df"""
    # Get target data
    target_data = target_df[[key_col, 'target']].copy()  # Fixed: double brackets for column selection
    target_data = target_data.rename(columns={'target': 'target_next_year'})
    
    # Merge and keep only matching keys
    lagged_df = features_df.merge(target_data, on=key_col, how='inner')
    
    # Replace original target if it exists
    if 'target' in lagged_df.columns:
        lagged_df = lagged_df.drop('target', axis=1)
    
    lagged_df = lagged_df.rename(columns={'target_next_year': 'target'})
    
    return lagged_df

# Apply the function
df_2018_lagged = create_lagged_dataset(df_2018, df_2019)
df_2019_lagged = create_lagged_dataset(df_2019, df_2020)

In [18]:
def optimize_dtypes(df, exclude_cols=['target']):
    """Convert float to float32 and int to int8/int16 based on range, excluding specified columns"""
    df_optimized = df.copy()
    
    # Get columns to convert (exclude specified columns)
    cols_to_convert = [col for col in df_optimized.columns if col not in exclude_cols]
    
    # Convert float columns to float32 (Parquet compatible)
    float_cols = df_optimized[cols_to_convert].select_dtypes(include=['float64']).columns
    df_optimized[float_cols] = df_optimized[float_cols].astype('float32')
    
    # Convert int columns
    int_cols = df_optimized[cols_to_convert].select_dtypes(include=['int64', 'int32']).columns
    for col in int_cols:
        col_values = df_optimized[col].dropna()
        if len(col_values) > 0:
            min_val = col_values.min()
            max_val = col_values.max()
            
            if min_val >= -128 and max_val <= 127:
                df_optimized[col] = df_optimized[col].astype('int8')
            elif min_val >= -32768 and max_val <= 32767:
                df_optimized[col] = df_optimized[col].astype('int16')
    
    return df_optimized

In [19]:
df_2018_lagged.to_parquet("df_2018_lagged_target.parquet")
df_2019_lagged.to_parquet("df_2019_lagged_target.parquet")

In [9]:
df_2018_lagged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2921347 entries, 0 to 2921346
Columns: 141 entries, UNIQUE_CONSUMER_KEY to target
dtypes: float32(34), float64(104), int64(1), object(1), uint8(1)
memory usage: 2.7+ GB


In [23]:
df_2018_lagged = pd.read_parquet("df_2018_lagged_target.parquet")
df_2019_lagged = pd.read_parquet("df_2018_lagged_target.parquet")

In [24]:
df_2018_lagged = optimize_dtypes(df_2018_lagged)
df_2019_lagged = optimize_dtypes(df_2018_lagged)

In [25]:
df_2018_lagged.to_parquet("df_2018_lagged_target.parquet")
df_2019_lagged.to_parquet("df_2019_lagged_target.parquet")

In [26]:
import gc

In [27]:
gc.collect()

959

In [ ]:
# #CRP02_0206015060: Number of Trades Major Derogatory Occurred in 12 Months  
# # The column you want to check for
# col_to_check = 'CRP02_0206015060'

# # 1. Check if the column exists in the DataFrame
# if col_to_check in final_df.columns:
#     print(f"Found column '{col_to_check}'. Processing...")
#     null_count = final_df[col_to_check].isnull().sum()
#     print(f"Total null values in '{col_to_check}': {null_count}")
#     print("-" * 30)
#     final_df[col_to_check] = final_df[col_to_check].fillna(0).astype('int8')

#     # 2. Create the 'target' column

#     final_df['target'] = np.where(final_df[col_to_check] > 0, 1, 0).astype('uint8')

#     # 3. Drop the original column
#     # final_df = final_df.drop(columns=[col_to_check], inplace=True)
#     del final_df[col_to_check]

#     print("\ntarget created")

# else:
#     print(f"Column '{col_to_check}' not found in the DataFrame.")
# gc.collect()

In [ ]:
# # 1. Calculate the percentage of missing values (not mem efficent)
# missing_percentage = final_df.isnull().sum() / len(final_df) * 100

# # 2. Filter out columns with no missing values and sort the result
# missing_percentage = missing_percentage[missing_percentage > 0].sort_values()

# # 3. Generate the plot
# plt.figure(figsize=(10, 8))
# missing_percentage.plot(kind='barh', color='skyblue')
# plt.title('% of Missing Values per Column')
# plt.xlabel('Percentage Missing (%)')
# plt.ylabel('Columns')
# # Add percentage labels to the bars
# for index, value in enumerate(missing_percentage):
#     plt.text(value, index, f' {value:.2f}%')

# plt.tight_layout()


In [ ]:
# # 1. Calculate the percentage of missing values efficiently
# total_rows = len(final_df)
# missing_data = {}
# for col in final_df.columns:
#     missing_count = final_df[col].isnull().sum()
#     if missing_count > 0:
#         missing_data[col] = (missing_count / total_rows) * 100

# # Convert the dictionary to a sorted pandas Series
# missing_percentage = pd.Series(missing_data).sort_values()

# # 3. Generate the plot (No changes needed here)
# plt.figure(figsize=(10, 8))
# missing_percentage.plot(kind='barh', color='skyblue')
# plt.title('% of Missing Values per Column')
# plt.xlabel('Percentage Missing (%)')
# plt.ylabel('Columns')

# # Add percentage labels to the bars
# for index, value in enumerate(missing_percentage):
#     plt.text(value, index, f' {value:.2f}%')

# plt.tight_layout()
# plt.show()
# gc.collect()

In [ ]:
# columns_to_drop = missing_percentage[missing_percentage > 30].index.tolist()
# columns_to_keep = missing_percentage[missing_percentage <= 30].index.tolist()+ ['CRP02_0206015060']
# #del final_df
# df_cleaned = dd.read_parquet('/gpfs/user_home/os_home_dirs/zsherif2/test/efx2018_2019_cleaned.parquet', columns=columns_to_keep)

# print("\nLoad complete.")
# print("DataFrame shape:", df_cleaned.shape)

In [ ]:
# # #not mem efficent
# columns_to_drop = missing_percentage[missing_percentage > 30].index.tolist()

# # 3. Drop the identified columns
# df_cleaned = final_df.drop(columns=columns_to_drop, inplace=True)

# print("\nColumns dropped:", columns_to_drop)
# print("New DataFrame shape:", df_cleaned.shape)

# del final_df
# gc.collect()

In [ ]:
# columns_to_keep = missing_percentage[missing_percentage <= 30].index.tolist()
# df_cleaned = final_df[columns_to_keep]
# del final_df  # Free up memory immediately

# print(f"Columns kept: {len(columns_to_keep)}")
# print("New DataFrame shape:", df_cleaned.shape)
# gc.collect()

In [13]:

df_cleaned=final_df

In [29]:
# train_df = df_cleaned[df_cleaned['META_REVISION'] == 201807].copy()
# test_df = df_cleaned[df_cleaned['META_REVISION'] == 201907].copy()
train_df = df_2018_lagged
test_df = df_2019_lagged

In [35]:
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(train_df.iloc[:, 2:140])


In [36]:
X_test_imputed = imputer.transform(test_df.iloc[:, 2:140])
all_features = train_df.columns.tolist()
features_to_impute = train_df.columns[2:140]

imputed_train_df = pd.DataFrame(X_train_imputed, columns=features_to_impute, index=train_df.index)
imputed_test_df = pd.DataFrame(X_test_imputed, columns=features_to_impute, index=test_df.index)

unimputed_train_df = train_df.iloc[:, :2]
unimputed_test_df = test_df.iloc[:, :2]
train_target = train_df['target']
test_target = test_df['target']

train_final_df = pd.concat([train_target, imputed_train_df], axis=1)
test_final_df = pd.concat([test_target, imputed_test_df], axis=1)

In [37]:
train_final_df.head()

,target,VANTAGE40,CRP02_0101010000,CRP02_0101110000,CRP02_0101120000,CRP02_0101290000,CRP02_0101470000,CRP02_0105010000,CRP02_0105030000,CRP02_0143010000,...,CRP02_0906575030,CRP02_0908015020,CRP02_0908015030,CRP02_0908015040,CRP02_0908015060,CRP02_0908475020,CRP02_0908585020,CRP02_0910015000,CRP02_0910035000,CRP02_0913015010
0,0,748.0,157.0,144.0,157.0,80.0,157.0,0.0,0.0,63.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1852,0.1923,0.3571
1,0,606.0,88.0,88.0,30.0,76.0,88.0,0.0,0.0,49.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.3030,0.3030,0.7083
2,0,559.0,33.0,163.0,169.0,140.0,33.0,0.0,0.0,136.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0000,1.0000
3,0,759.0,178.0,127.0,127.0,178.0,133.0,0.0,0.0,178.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0000,0.2000
4,0,810.0,322.0,322.0,322.0,136.0,322.0,0.0,0.0,200.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1481,0.1481,0.2143


In [38]:
# 2. Save each DataFrame to a Parquet file
print("Saving training data to train_imputed.parquet...")
train_final_df.to_parquet('train_imputed.parquet', index=False)

print("Saving testing data to test_imputed.parquet...")
test_final_df.to_parquet('test_imputed.parquet', index=False)

print("✅ Both files have been saved successfully.")

Saving training data to train_final.parquet...
Saving testing data to test_final.parquet...
✅ Both files have been saved successfully.


In [2]:
train_final_df = pd.read_parquet("train_imputed.parquet")
test_final_df = pd.read_parquet("test_imputed.parquet")

In [39]:
test_final_df.shape

(2921347, 139)

In [40]:
train_final_df.shape

(2921347, 139)

In [41]:
train_final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2921347 entries, 0 to 2921346
Columns: 139 entries, target to CRP02_0913015010
dtypes: float32(138), uint8(1)
memory usage: 1.5 GB


In [42]:
train_final_df.head()

,target,VANTAGE40,CRP02_0101010000,CRP02_0101110000,CRP02_0101120000,CRP02_0101290000,CRP02_0101470000,CRP02_0105010000,CRP02_0105030000,CRP02_0143010000,...,CRP02_0906575030,CRP02_0908015020,CRP02_0908015030,CRP02_0908015040,CRP02_0908015060,CRP02_0908475020,CRP02_0908585020,CRP02_0910015000,CRP02_0910035000,CRP02_0913015010
0,0,748.0,157.0,144.0,157.0,80.0,157.0,0.0,0.0,63.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1852,0.1923,0.3571
1,0,606.0,88.0,88.0,30.0,76.0,88.0,0.0,0.0,49.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.3030,0.3030,0.7083
2,0,559.0,33.0,163.0,169.0,140.0,33.0,0.0,0.0,136.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0000,1.0000
3,0,759.0,178.0,127.0,127.0,178.0,133.0,0.0,0.0,178.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0000,0.2000
4,0,810.0,322.0,322.0,322.0,136.0,322.0,0.0,0.0,200.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1481,0.1481,0.2143


In [3]:
features = train_final_df.columns[1:].tolist()
print(features)

['VANTAGE40', 'CRP02_0101010000', 'CRP02_0101110000', 'CRP02_0101120000', 'CRP02_0101290000', 'CRP02_0101470000', 'CRP02_0105010000', 'CRP02_0105030000', 'CRP02_0143010000', 'CRP02_0205017500', 'CRP02_0211015021', 'CRP02_0211015031', 'CRP02_0211015041', 'CRP02_0211117061', 'CRP02_0211118061', 'CRP02_0211125021', 'CRP02_0211125031', 'CRP02_0211125041', 'CRP02_0211127061', 'CRP02_0211128061', 'CRP02_0211295021', 'CRP02_0211295031', 'CRP02_0211295041', 'CRP02_0211448061', 'CRP02_0211475021', 'CRP02_0211475031', 'CRP02_0211475041', 'CRP02_0212035011', 'CRP02_0212125031', 'CRP02_0212125041', 'CRP02_0212535011', 'CRP02_0212575011', 'CRP02_0212015022', 'CRP02_0212015032', 'CRP02_0212015042', 'CRP02_0212465042', 'CRP02_0213015020', 'CRP02_0213015030', 'CRP02_0213015040', 'CRP02_0213017050', 'CRP02_0213125020', 'CRP02_0213127050', 'CRP02_0213295020', 'CRP02_0213295030', 'CRP02_0213297050', 'CRP02_0213445030', 'CRP02_0213465020', 'CRP02_0213475020', 'CRP02_0213475030', 'CRP02_0213477050', 'CRP02

In [ ]:
 #X_train_final = train_final_df.drop(columns = ['UNIQUE_CONSUMER_KEY','META_REVISION'])

In [19]:
# pip install varclushi

Note: you may need to restart the kernel to use updated packages.


In [45]:
# del final_df
# del df_cleaned
del train_df
del test_df
del imputer
del X_train_imputed
del X_test_imputed
del imputed_train_df
del imputed_test_df
del unimputed_train_df
del unimputed_test_df

In [48]:
from sklearn.model_selection import train_test_split

# Get stratified sample of 300,000 observations
stratified_sample, _ = train_test_split(
    train_final_df, 
    train_size=300000,
    stratify=train_final_df['target'],  # replace with your actual target column name
    random_state=42
)

# Verify all columns are preserved
print("Original columns:", train_final_df.columns.tolist())
print("Sample columns:", stratified_sample.columns.tolist())
print("Columns match:", train_final_df.columns.equals(stratified_sample.columns))
print("Sample shape:", stratified_sample.shape)

Original columns: ['target', 'VANTAGE40', 'CRP02_0101010000', 'CRP02_0101110000', 'CRP02_0101120000', 'CRP02_0101290000', 'CRP02_0101470000', 'CRP02_0105010000', 'CRP02_0105030000', 'CRP02_0143010000', 'CRP02_0205017500', 'CRP02_0211015021', 'CRP02_0211015031', 'CRP02_0211015041', 'CRP02_0211117061', 'CRP02_0211118061', 'CRP02_0211125021', 'CRP02_0211125031', 'CRP02_0211125041', 'CRP02_0211127061', 'CRP02_0211128061', 'CRP02_0211295021', 'CRP02_0211295031', 'CRP02_0211295041', 'CRP02_0211448061', 'CRP02_0211475021', 'CRP02_0211475031', 'CRP02_0211475041', 'CRP02_0212035011', 'CRP02_0212125031', 'CRP02_0212125041', 'CRP02_0212535011', 'CRP02_0212575011', 'CRP02_0212015022', 'CRP02_0212015032', 'CRP02_0212015042', 'CRP02_0212465042', 'CRP02_0213015020', 'CRP02_0213015030', 'CRP02_0213015040', 'CRP02_0213017050', 'CRP02_0213125020', 'CRP02_0213127050', 'CRP02_0213295020', 'CRP02_0213295030', 'CRP02_0213297050', 'CRP02_0213445030', 'CRP02_0213465020', 'CRP02_0213475020', 'CRP02_0213475030'

In [10]:
from varclushi import VarClusHi

# # 1. Get the full list of row indices from the original dataframe
# full_indices = train_final_df.index

# # 2. Choose 100,000 random indices from the list
# # Using np.random.choice is efficient for this
# sample_indices = np.random.choice(full_indices, size=100000, replace=False)

# # 3. Select the rows AND columns in a single operation
# # This avoids creating the large intermediate copy
# sample_df = train_final_df.loc[sample_indices, features]

# # Now you can proceed as before without memory errors
# sample_df = sample_df.astype('float32')
vc = VarClusHi(df=train_final_df, maxeigval2=0.7,feat_list=features )
vc.varclus()

cluster_report_df = vc.rsquare


In [11]:
vc.info

,Cluster,N_Vars,Eigval1,Eigval2,VarProp
0,0,7,5.827330,0.644150,0.832476
1,1,2,1.717504,0.282496,0.858752
2,2,7,5.585259,0.605626,0.797894
3,3,5,4.140398,0.604396,0.828080
4,4,4,3.315496,0.618344,0.828874
5,5,5,4.013539,0.390298,0.802708
6,6,4,2.782226,0.690902,0.695557
7,7,4,3.368830,0.501724,0.842208
8,8,3,2.143174,0.544474,0.714391
9,9,4,3.924419,0.063970,0.981105


In [12]:
print(cluster_report_df)

     Cluster          Variable    RS_Own     RS_NC      RS_Ratio
0          0  CRP02_0903015000  0.468428  0.334110  7.982880e-01
1          0  CRP02_0905014000  0.852563  0.468789  2.775492e-01
2          0  CRP02_0905015000  0.873862  0.473563  2.396065e-01
3          0  CRP02_0908015020  0.931374  0.494709  1.358144e-01
4          0  CRP02_0908015030  0.941312  0.452685  1.072294e-01
..       ...               ...       ...       ...           ...
133       51  CRP02_0404478001  1.000000  0.080679  0.000000e+00
134       52  CRP02_0904572000  1.000000  0.345954  0.000000e+00
135       53  CRP02_0211295031  1.000000  0.250815  0.000000e+00
136       54  CRP02_0212125041  1.000000  0.179332  0.000000e+00
137       55  CRP02_0212125031  1.000000  0.212297  2.818887e-16

[138 rows x 5 columns]


In [13]:
# Get the index of the row with the minimum RS_Ratio for each cluster
idx = cluster_report_df.groupby('Cluster')['RS_Ratio'].idxmin()

# Select the corresponding rows to find the best feature for each cluster
selected_features_df = cluster_report_df.loc[idx]

# Get the list of selected feature names
final_feature_list = selected_features_df['Variable'].tolist()

print("Selected Features:")
print(final_feature_list)

Selected Features:
['CRP02_0908015030', 'CRP02_0906128010', 'CRP02_0231017000', 'CRP02_0245035000', 'CRP02_0213465020', 'CRP02_0101470000', 'CRP02_0232017050', 'CRP02_0504128000', 'CRP02_0211125031', 'CRP02_0501474000', 'CRP02_0211127061', 'CRP02_0211015041', 'CRP02_0211295021', 'CRP02_0402012001', 'CRP02_0212015042', 'CRP02_0214465000', 'CRP02_0906298010', 'CRP02_0905475000', 'CRP02_0211448061', 'CRP02_0228170000', 'CRP02_0211475041', 'CRP02_0910015000', 'CRP02_0105030000', 'CRP02_0213127050', 'CRP02_0232012030', 'CRP02_0901475000', 'CRP02_0214294000', 'CRP02_0213295020', 'CRP02_0404588001', 'CRP02_0213017050', 'CRP02_0211125021', 'CRP02_0212535011', 'CRP02_0213475030', 'CRP02_0801015000', 'CRP02_0211118061', 'CRP02_0217010000', 'CRP02_0228150000', 'CRP02_0504472000', 'CRP02_0213295030', 'CRP02_0228160000', 'CRP02_0913015010', 'CRP02_0502122003', 'CRP02_0232475050', 'CRP02_0903588000', 'CRP02_0401448000', 'CRP02_0205017500', 'CRP02_0214012000', 'CRP02_0213015040', 'CRP02_0232015020', 

In [14]:
train_selected_df= train_final_df[final_feature_list]
test_selected_df = test_final_df[final_feature_list]

In [53]:
del vc

In [54]:
# import pandas as pd
# import numpy as np
# from scipy.cluster import hierarchy
# from scipy.spatial.distance import squareform
# from statsmodels.stats.outliers_influence import variance_inflation_factor
# from statsmodels.tools.tools import add_constant
# import matplotlib.pyplot as plt

# # 2. Perform Hierarchical Clustering
# corr_matrix = X_train_final.corr()
# # Convert correlation to distance (1 - |r|)
# dist_matrix = 1 - np.abs(corr_matrix)
# # Convert to a condensed distance matrix for the linkage function
# condensed_dist = squareform(dist_matrix)

# # Perform clustering
# Z = hierarchy.linkage(condensed_dist, method='average')


# plt.figure(figsize=(10, 5))
# plt.title('Hierarchical Clustering Dendrogram')
# plt.xlabel('Features')
# plt.ylabel('Distance (1 - |correlation|)')
# hierarchy.dendrogram(Z, labels=X_train_final.columns, leaf_rotation=90)
# plt.axhline(y=0.4, c='k', ls='--') # Add a cutoff line
# plt.show()

# # 3. Form Flat Clusters
# # Use a distance threshold to form clusters. A lower threshold means more clusters.
# # A threshold of 0.4 means features in a cluster have at least ~0.6 correlation.
# distance_threshold = 0.4
# cluster_labels = hierarchy.fcluster(Z, distance_threshold, criterion='distance')

# # Create a mapping of cluster ID to feature names
# clusters = {}
# for i, feature in enumerate(X_train_final.columns):
#     cluster_id = cluster_labels[i]
#     if cluster_id not in clusters:
#         clusters[cluster_id] = []
#     clusters[cluster_id].append(feature)

# print("--- Formed Clusters ---")
# print(clusters)

# # 4. Select the Best Feature from Each Cluster using VIF
# final_features = []
# for cluster_id, features in clusters.items():
#     if len(features) == 1:
#         # If the cluster has only one feature, keep it
#         final_features.append(features[0])
#     else:
#         # For clusters with multiple features, use VIF
#         X_cluster = X_train_final[features]
#         X_cluster_const = add_constant(X_cluster)
        
#         # Calculate VIF for each feature in the cluster
#         vif = pd.Series(
#             [variance_inflation_factor(X_cluster_const.values, i) for i in range(X_cluster_const.shape[1])],
#             index=X_cluster_const.columns
#         ).drop('const')
        
#         # Select the feature with the lowest VIF
#         best_feature = vif.idxmin()
#         final_features.append(best_feature)
#         print(f"\nCluster {cluster_id} (Features: {features}):")
#         print(f"VIF Scores:\n{vif}")
#         print(f"Selected: '{best_feature}' (Lowest VIF)")

# print("\n--- Final Selected Features ---")
# print(final_features)

# # Now you can create your final training and testing sets
# # X_train_selected = X_train_final[final_features]
# # X_test_selected = X_test_final[final_features] # Assuming X_test_final exists

In [ ]:
# cols_to_keep= ['UNIQUE_CONSUMER_KEY','META_REVISION'] + final_features

In [ ]:
# from xgboost import XGBClassifier

# # 3. Initialize and train the Gradient Boosting model
# # You can tune these parameters (n_estimators, max_depth, etc.) for better performance
# gbm = XGBClassifier(
#     n_estimators=100,    # number of boosted trees
#     learning_rate=0.1,   # step size shrinkage
#     max_depth=3,         # max depth of each tree
#     random_state=42,
#     n_jobs=-1            # use all CPU cores for training
# )
# # Fit the model on the training data
# gbm.fit(train_final_df[features], train_final_df['target'])

# # 4. Get and display feature importances
# feature_importances = gbm.feature_importances_

# # Create a pandas DataFrame for better visualization
# importance_df = pd.DataFrame({
#     'Feature': features,
#     'Importance': feature_importances
# })

# # Sort the features by importance in descending order
# importance_df = importance_df.sort_values(by='Importance', ascending=False)

# # Display the ranked features
# print("Feature Importances from GBM:")
# print(importance_df)


In [ ]:
# top_k = 30
# selected_features = importance_df['Feature'].head(top_k).tolist()

# print(f"\nTop {top_k} selected features:")
# print(selected_features)

In [ ]:
# for feature in final_feature_list:
#     if feature in selected_features:
#         print(feature)

In [ ]:
# train_df = train_selected_df
# test_df = test_selected_df

In [ ]:
# train_df.shape
# test_df.shape

In [ ]:
# from sklearn.base import BaseEstimator, TransformerMixin
# from sklearn.compose import ColumnTransformer

# class OutlierCapper(BaseEstimator, TransformerMixin):
#     def __init__(self, iqr_multiplier=1.5):
#         self.iqr_multiplier = iqr_multiplier
#         self.lower_bounds_ = {}
#         self.upper_bounds_ = {}

#     def fit(self, X, y=None):
#         X_ = pd.DataFrame(X)
#         for col in X_.columns:
#             q1 = X_[col].quantile(0.25)
#             q3 = X_[col].quantile(0.75)
#             iqr = q3 - q1
#             self.lower_bounds_[col] = q1 - self.iqr_multiplier * iqr
#             self.upper_bounds_[col] = q3 + self.iqr_multiplier * iqr
#         return self

#     def transform(self, X):
#         X_capped = pd.DataFrame(X).copy()
#         for col in X_capped.columns:
#             lower = self.lower_bounds_.get(col)
#             upper = self.upper_bounds_.get(col)
#             X_capped[col] = X_capped[col].clip(lower, upper)
#         return X_capped



In [ ]:
# # non_feature_cols = ['UNIQUE_CONSUMER_KEY', 'META_REVISION']
# # feature_cols = [col for col in train_df.columns if col not in non_feature_cols]

# # 4. Separate the feature data for processing
# X_train_features = train_df
# X_test_features = test_df

# # 5. Initialize the capper, fit on training features, and transform both sets
# capper = OutlierCapper()
# capper.fit(X_train_features)

# train_capped_features = capper.transform(X_train_features)
# test_capped_features = capper.transform(X_test_features)

# # 6. Join the processed features back with the non-feature columns
# # train_processed_df = pd.concat([
# #     train_df[non_feature_cols].reset_index(drop=True),
# #     train_capped_features.reset_index(drop=True)
# # ], axis=1)

# # test_processed_df = pd.concat([
# #     test_df[non_feature_cols].reset_index(drop=True),
# #     test_capped_features.reset_index(drop=True)
# # ], axis=1)



In [57]:
from typing import List, Optional

def cap_chebyshev_outliers(df: pd.DataFrame, columns: Optional[List[str]] = None, rarity_threshold: int = 2_000_000):
    """
    Identifies and caps outliers in a DataFrame using Chebyshev's inequality.

    Args:
        df: The pandas DataFrame.
        columns: A list of column names to process. If None, all numerical 
                 columns will be processed.
        rarity_threshold: Defines the rarity for an outlier (e.g., 1 in 2,000,000).

    Returns:
        A new pandas DataFrame with the outliers in the specified columns capped.
    """
    # Create a copy to avoid modifying the original DataFrame
    df_capped = df.copy()

    # If no columns are specified, use all numerical columns
    if columns is None:
        columns = df.select_dtypes(include=np.number).columns.tolist()
    
    print(f"Capping outliers with a rarity threshold of 1 in {rarity_threshold:,}...\n")
    
    # Iterate over each specified column
    for column in columns:
        if column not in df_capped.columns:
            print(f"Warning: Column '{column}' not in DataFrame. Skipping.")
            continue
        
        # Calculate k, mean, std, and the outlier thresholds
        proportion = 1.0 / rarity_threshold
        k = np.sqrt(1.0 / proportion)
        mean = df_capped[column].mean()
        std_dev = df_capped[column].std()
        lower_bound = mean - k * std_dev
        upper_bound = mean + k * std_dev
        
        # Count how many values will be capped
        outliers_count = df_capped[(df_capped[column] < lower_bound) | (df_capped[column] > upper_bound)].shape[0]
        
        # Use clip() to cap the values at the calculated bounds
        df_capped[column] = df_capped[column].clip(lower=lower_bound, upper=upper_bound)
        
        print(f"Column '{column}': Capped {outliers_count} outliers (Bounds: {lower_bound:.2f} to {upper_bound:.2f})")

    return df_capped


In [58]:
train_chebyshev_df = cap_chebyshev_outliers(train_selected_df, rarity_threshold=100000)
test_chebyshev_df = cap_chebyshev_outliers(test_selected_df, rarity_threshold=100000)

Capping outliers with a rarity threshold of 1 in 100,000...

Column 'CRP02_0908015030': Capped 0 outliers (Bounds: -85.37 to 85.59)
Column 'CRP02_0906128010': Capped 0 outliers (Bounds: -98.76 to 99.51)
Column 'CRP02_0213465020': Capped 0 outliers (Bounds: -245.76 to 245.93)
Column 'CRP02_0214015000': Capped 0 outliers (Bounds: -208.31 to 208.81)
Column 'CRP02_0231015000': Capped 0 outliers (Bounds: -216.86 to 217.06)
Column 'CRP02_0101470000': Capped 0 outliers (Bounds: -40864.52 to 41285.54)
Column 'CRP02_0504538000': Capped 1 outliers (Bounds: -4670788.37 to 4681289.59)
Column 'CRP02_0232012030': Capped 0 outliers (Bounds: -43.13 to 43.16)
Column 'CRP02_0501474000': Capped 0 outliers (Bounds: -93.10 to 93.62)
Column 'CRP02_0211128061': Capped 0 outliers (Bounds: -69.06 to 69.15)
Column 'CRP02_0211015041': Capped 0 outliers (Bounds: -62.60 to 62.62)
Column 'CRP02_0212015042': Capped 0 outliers (Bounds: -31.80 to 31.82)
Column 'CRP02_0232015050': Capped 0 outliers (Bounds: -172.88 to 

In [41]:
#before capping
train_selected_df.skew()

CRP02_0908015030     2.511032
CRP02_0501467000     1.966850
CRP02_0214015000    13.096768
CRP02_0231015000    19.546686
CRP02_0213465020    20.888138
                      ...    
CRP02_0903448000     0.782906
CRP02_0212465042    19.060564
CRP02_0906575030    20.003399
CRP02_0905128000    -2.435010
CRP02_0232012050    35.746582
Length: 75, dtype: float32

In [ ]:
# after capping with 1.5IQR
# train_capped_features.skew()

In [43]:
#after capping with chebyshev
train_chebyshev_df.skew()

CRP02_0908015030     2.511032
CRP02_0501467000     1.966850
CRP02_0214015000    13.096768
CRP02_0231015000    19.546686
CRP02_0213465020    20.888138
                      ...    
CRP02_0903448000     0.782906
CRP02_0212465042    19.060564
CRP02_0906575030    20.003399
CRP02_0905128000    -2.435010
CRP02_0232012050    35.746582
Length: 75, dtype: float64

In [49]:
del train_chebyshev_df, test_chebyshev_df

In [103]:
gc.collect()

6008

In [15]:
train_selected_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2921347 entries, 0 to 2921346
Data columns (total 56 columns):
 #   Column            Dtype  
---  ------            -----  
 0   CRP02_0908015030  float32
 1   CRP02_0906128010  float32
 2   CRP02_0231017000  float32
 3   CRP02_0245035000  float32
 4   CRP02_0213465020  float32
 5   CRP02_0101470000  float32
 6   CRP02_0232017050  float32
 7   CRP02_0504128000  float32
 8   CRP02_0211125031  float32
 9   CRP02_0501474000  float32
 10  CRP02_0211127061  float32
 11  CRP02_0211015041  float32
 12  CRP02_0211295021  float32
 13  CRP02_0402012001  float32
 14  CRP02_0212015042  float32
 15  CRP02_0214465000  float32
 16  CRP02_0906298010  float32
 17  CRP02_0905475000  float32
 18  CRP02_0211448061  float32
 19  CRP02_0228170000  float32
 20  CRP02_0211475041  float32
 21  CRP02_0910015000  float32
 22  CRP02_0105030000  float32
 23  CRP02_0213127050  float32
 24  CRP02_0232012030  float32
 25  CRP02_0901475000  float32
 26  CRP02_02142940

In [16]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

def calculate_iterative_vif(
    df: pd.DataFrame, 
    threshold: float = 10.0, 
    sample_size: int = 300000
) -> pd.DataFrame:
    """
    Iteratively calculates VIF and removes features with VIF above a threshold.

    To prevent MemoryError on large datasets, VIF is calculated on a random
    sample of the data.

    Args:
        df (pd.DataFrame): The DataFrame containing the features.
        threshold (float): The VIF threshold for feature elimination.
        sample_size (int): The number of rows to sample for VIF calculation.

    Returns:
        pd.DataFrame: A DataFrame with the final features and their VIF scores.
    """
    # --- NEW: Sample the data to prevent MemoryError ---
    # if len(df) > sample_size:
    #     print(f"Dataset is large. Using a random sample of {sample_size} rows for VIF calculation.")
    #     df_sample = df.sample(n=sample_size, random_state=42)
    # else:
    #     df_sample = df
    
    df_sample = df    
    # Create a copy to avoid modifying the sampled DataFrame
    X = df_sample.copy()
    
    while True:
        # Add a constant for VIF calculation
        X_with_const = sm.add_constant(X)
        
        # Create a DataFrame to store VIF scores
        vif_df = pd.DataFrame()
        vif_df["feature"] = X_with_const.columns
        vif_df["vif"] = [
            variance_inflation_factor(X_with_const.values, i)
            for i in range(X_with_const.shape[1])
        ]
        
        # Filter out the 'const' row
        vif_features = vif_df[vif_df['feature'] != 'const']
        
        # Find the maximum VIF
        max_vif = vif_features['vif'].max()
        
        # If the max VIF is below the threshold, we're done
        if max_vif <= threshold:
            break
            
        # Otherwise, find the feature with the highest VIF and remove it
        feature_to_remove = vif_features.sort_values('vif', ascending=False)['feature'].iloc[0]
        print(f"Removing '{feature_to_remove}' with VIF: {max_vif:.2f}")
        X = X.drop(columns=[feature_to_remove])

    print("\nFinal features after VIF selection:")
    # Return the VIF scores for the final set of columns
    # Also return the list of final columns to easily filter the original df
    final_features = X.columns.tolist()
    return vif_features.sort_values('vif', ascending=False).reset_index(drop=True), final_features

In [17]:
# Run the iterative VIF calculation
final_vif_df, final_features = calculate_iterative_vif(train_selected_df, threshold=10.0)
    
# Print the final results
final_vif_df


Final features after VIF selection:


,feature,vif
0,VANTAGE40,2.679861
1,CRP02_0908015030,2.300666
2,CRP02_0906128010,2.086025
3,CRP02_0904572000,2.040439
4,CRP02_0905475000,2.005835
5,CRP02_0903588000,1.978615
6,CRP02_0501474000,1.956788
7,CRP02_0213475030,1.913135
8,CRP02_0214465000,1.807380
9,CRP02_0232017050,1.805544


In [64]:
gc.collect()

4254

In [1]:
%whos

Interactive namespace is empty.


In [113]:
del sample_df, idx, cluster_report_df  

In [18]:
train_selected_df=train_selected_df[final_features]
test_selected_df=test_selected_df[final_features]

In [19]:
import pandas as pd

# Assuming train_selected_df, test_selected_df, train_final_df, 
# and test_final_df are all pre-existing pandas DataFrames.

# --- Process Training Data ---
print("⚙️ Processing training data...")
# 1. Concatenate the 'target' column
final_train_df = pd.concat([train_selected_df, train_final_df['target']], axis=1)

# 2. Save the new DataFrame to a Parquet file
print("💾 Saving training data...")
final_train_df.to_parquet('train_selected_final.parquet', index=False)


# --- Process Testing Data ---
print("\n⚙️ Processing testing data...")
# 1. Concatenate the 'target' column
final_test_df = pd.concat([test_selected_df, test_final_df['target']], axis=1)

# 2. Save the new DataFrame to a Parquet file
print("💾 Saving testing data...")
final_test_df.to_parquet('test_selected_final.parquet', index=False)

print("\n✅ Files saved successfully.")

⚙️ Processing training data...
💾 Saving training data...

⚙️ Processing testing data...
💾 Saving testing data...

✅ Files saved successfully.


In [20]:
import pandas as pd
import os

# --- Setup ---
# Specify the same output path you used to save the files.
output_path = '/gpfs/user_home/os_home_dirs/zsherif2/test/' # Example path

# Define the file paths
train_file_path = os.path.join(output_path, 'train_selected_final.parquet')
test_file_path = os.path.join(output_path, 'test_selected_final.parquet')

# --- Read the Parquet files ---
try:
    # Load the training data
    train_df = pd.read_parquet(train_file_path)
    print("Successfully loaded 'train_selected_final.parquet'")
    print("Train DataFrame shape:", train_df.shape)
    print("First 5 rows of training data:")
    print(train_df.head())
    print("-" * 50)

    # Load the testing data
    test_df = pd.read_parquet(test_file_path)
    print("Successfully loaded 'test_selected_final.parquet'")
    print("Test DataFrame shape:", test_df.shape)
    print("First 5 rows of testing data:")
    print(test_df.head())

except FileNotFoundError as e:
    print(f"Error: Could not find the file. Make sure the path is correct.")
    print(e)
except Exception as e:
    print(f"An error occurred: {e}")

Successfully loaded 'train_selected_final.parquet'
Train DataFrame shape: (2921347, 57)
First 5 rows of training data:
   CRP02_0908015030  CRP02_0906128010  CRP02_0231017000  CRP02_0245035000  \
0               0.0            0.1111               0.0               0.0   
1               0.0            1.0000               0.0               0.0   
2               0.0            0.3333               0.0               0.0   
3               0.0            0.6667               0.0               0.0   
4               0.0            0.1667               0.0               0.0   

   CRP02_0213465020  CRP02_0101470000  CRP02_0232017050  CRP02_0504128000  \
0               0.0             157.0               0.0            3015.0   
1               0.0              88.0               0.0              56.0   
2               0.0              33.0               0.0            4030.0   
3               0.0             133.0               0.0            2984.0   
4               0.0             3

In [21]:
final_train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2921347 entries, 0 to 2921346
Data columns (total 57 columns):
 #   Column            Dtype  
---  ------            -----  
 0   CRP02_0908015030  float32
 1   CRP02_0906128010  float32
 2   CRP02_0231017000  float32
 3   CRP02_0245035000  float32
 4   CRP02_0213465020  float32
 5   CRP02_0101470000  float32
 6   CRP02_0232017050  float32
 7   CRP02_0504128000  float32
 8   CRP02_0211125031  float32
 9   CRP02_0501474000  float32
 10  CRP02_0211127061  float32
 11  CRP02_0211015041  float32
 12  CRP02_0211295021  float32
 13  CRP02_0402012001  float32
 14  CRP02_0212015042  float32
 15  CRP02_0214465000  float32
 16  CRP02_0906298010  float32
 17  CRP02_0905475000  float32
 18  CRP02_0211448061  float32
 19  CRP02_0228170000  float32
 20  CRP02_0211475041  float32
 21  CRP02_0910015000  float32
 22  CRP02_0105030000  float32
 23  CRP02_0213127050  float32
 24  CRP02_0232012030  float32
 25  CRP02_0901475000  float32
 26  CRP02_02142940

In [22]:
final_test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2921347 entries, 0 to 2921346
Data columns (total 57 columns):
 #   Column            Dtype  
---  ------            -----  
 0   CRP02_0908015030  float32
 1   CRP02_0906128010  float32
 2   CRP02_0231017000  float32
 3   CRP02_0245035000  float32
 4   CRP02_0213465020  float32
 5   CRP02_0101470000  float32
 6   CRP02_0232017050  float32
 7   CRP02_0504128000  float32
 8   CRP02_0211125031  float32
 9   CRP02_0501474000  float32
 10  CRP02_0211127061  float32
 11  CRP02_0211015041  float32
 12  CRP02_0211295021  float32
 13  CRP02_0402012001  float32
 14  CRP02_0212015042  float32
 15  CRP02_0214465000  float32
 16  CRP02_0906298010  float32
 17  CRP02_0905475000  float32
 18  CRP02_0211448061  float32
 19  CRP02_0228170000  float32
 20  CRP02_0211475041  float32
 21  CRP02_0910015000  float32
 22  CRP02_0105030000  float32
 23  CRP02_0213127050  float32
 24  CRP02_0232012030  float32
 25  CRP02_0901475000  float32
 26  CRP02_02142940

In [98]:
del train_final_df, test_final_df

In [32]:
!pip install feature-engine

Defaulting to user installation because normal site-packages is not writeable


In [71]:
# import pandas as pd
# import numpy as np
# from sklearn.preprocessing import PowerTransformer
# from typing import Tuple
# import os
# import gc

# # You might need to install these libraries
# # !pip install feature-engine
# # !pip install pyarrow

# def apply_and_save_transformations(
#     train_df: pd.DataFrame,
#     test_df: pd.DataFrame,
#     output_path: str,
#     target_series: pd.Series = None
# ) -> None:
#     """
#     Applies a suite of transformations, saves each result to a Parquet file,
#     and clears it from memory to prevent MemoryErrors.

#     Args:
#         train_df (pd.DataFrame): The training dataframe with selected features.
#         test_df (pd.DataFrame): The testing dataframe with selected features.
#         output_path (str): The directory path to save the output Parquet files.
#         target_series (pd.Series): The binary target series required for
#                                    supervised transformations.
#     """
#     # --- Setup ---
#     # Create the output directory if it doesn't exist
#     os.makedirs(output_path, exist_ok=True)
    
#     numerical_cols = train_df.select_dtypes(include=np.number).columns.tolist()
#     epsilon = 1e-6  # To prevent division by zero or log(0)

#     # # --- 1. Log Transform ---
#     print("Applying Log transformation...")
#     train_log = train_df.copy()
#     test_log = test_df.copy()
#     for col in numerical_cols:
#         if (train_log[col] >= 0).all():
#             train_log[col] = np.log1p(train_log[col])
#             test_log[col] = np.log1p(test_log[col])
#     print("  - Saving transformed data to Parquet...")
#     train_log.to_parquet(os.path.join(output_path, 'train_log.parquet'))
#     test_log.to_parquet(os.path.join(output_path, 'test_log.parquet'))
#     del train_log, test_log
#     gc.collect()
#     print("✓ Log transformation complete and memory freed.\n")

#     # # --- 2. Square Transform ---
#     print("Applying Square transformation...")
#     train_sq = train_df.copy()
#     test_sq = test_df.copy()
#     train_sq[numerical_cols] = train_sq[numerical_cols]**2
#     test_sq[numerical_cols] = test_sq[numerical_cols]**2
#     print("  - Saving transformed data to Parquet...")
#     train_sq.to_parquet(os.path.join(output_path, 'train_square.parquet'))
#     test_sq.to_parquet(os.path.join(output_path, 'test_square.parquet'))
#     del train_sq, test_sq
#     gc.collect()
#     print("✓ Square transformation complete and memory freed.\n")

#     # # --- 3. Inverse Transform ---
#     print("Applying Inverse transformation...")
#     train_inv = train_df.copy()
#     test_inv = test_df.copy()
#     train_inv[numerical_cols] = 1 / (train_inv[numerical_cols] + epsilon)
#     test_inv[numerical_cols] = 1 / (test_inv[numerical_cols] + epsilon)
#     print("  - Saving transformed data to Parquet...")
#     train_inv.to_parquet(os.path.join(output_path, 'train_inverse.parquet'))
#     test_inv.to_parquet(os.path.join(output_path, 'test_inverse.parquet'))
#     del train_inv, test_inv
#     gc.collect()
#     print("✓ Inverse transformation complete and memory freed.\n")

#     # # --- 4. Log-Odds (Logit) Transform ---
#     print("Applying Log-Odds transformation...")
#     train_logodds = train_df.copy()
#     test_logodds = test_df.copy()
#     logodds_applied = False
#     for col in numerical_cols:
#         is_proportion = (train_logodds[col] > 0).all() and (train_logodds[col] < 1).all()
#         if is_proportion:
#             logodds_applied = True
#             p_train = np.clip(train_logodds[col], epsilon, 1 - epsilon)
#             p_test = np.clip(test_logodds[col], epsilon, 1 - epsilon)
#             train_logodds[col] = np.log(p_train / (1 - p_train))
#             test_logodds[col] = np.log(p_test / (1 - p_test))
    
#     if logodds_applied:
#         print("  - Saving transformed data to Parquet...")
#         train_logodds.to_parquet(os.path.join(output_path, 'train_logodds.parquet'))
#         test_logodds.to_parquet(os.path.join(output_path, 'test_logodds.parquet'))
#         print("✓ Log-Odds transformation complete and memory freed.\n")
#     else:
#         print("  - Skipping Log-Odds: No columns with values in (0, 1) range found.\n")
#     del train_logodds, test_logodds
#     gc.collect()

#     # --- 5. Yeo-Johnson Transform ---
#     print("Applying Yeo-Johnson transformation...")
#     if numerical_cols:
#         train_yj = train_df.copy()
#         test_yj = test_df.copy()
#         pt = PowerTransformer(method='yeo-johnson', standardize=True)
#         train_yj[numerical_cols] = pt.fit_transform(train_yj[numerical_cols])
#         test_yj[numerical_cols] = pt.transform(test_yj[numerical_cols])
#         print("  - Saving transformed data to Parquet...")
#         train_yj.to_parquet(os.path.join(output_path, 'train_yeo-johnson.parquet'))
#         test_yj.to_parquet(os.path.join(output_path, 'test_yeo-johnson.parquet'))
#         del train_yj, test_yj
#         gc.collect()
#         print("✓ Yeo-Johnson transformation complete and memory freed.\n")
#     else:
#         print("  - Skipping Yeo-Johnson: No numerical columns found.\n")

#     # --- 6. Supervised Transformations ---
#     if target_series is not None:
#         # --- Odds Ratio Transformation (for categorical features) ---
#         print("Applying Odds Ratio transformation...")
#         categorical_cols = train_df.select_dtypes(include=['object', 'category']).columns.tolist()

#         if not categorical_cols:
#             print("  - Skipping Odds Ratio: No categorical columns found.\n")
#         else:
#             train_or = train_df.copy()
#             test_or = test_df.copy()
#             temp_df = pd.concat([train_df, target_series], axis=1)
#             target_name = target_series.name

#             for col in categorical_cols:
#                 cross_tab = pd.crosstab(temp_df[col], temp_df[target_name])
#                 if 0 not in cross_tab.columns: cross_tab[0] = 0
#                 if 1 not in cross_tab.columns: cross_tab[1] = 0
#                 odds = (cross_tab[1] + epsilon) / (cross_tab[0] + epsilon)
#                 reference_category = train_df[col].mode()[0]
#                 reference_odds = odds.get(reference_category, epsilon)
#                 or_mapper = (odds / reference_odds).to_dict()
#                 train_or[col] = train_or[col].map(or_mapper).fillna(1.0)
#                 test_or[col] = test_or[col].map(or_mapper).fillna(1.0)

#             print("  - Saving transformed data to Parquet...")
#             train_or.to_parquet(os.path.join(output_path, 'train_odds_ratio.parquet'))
#             test_or.to_parquet(os.path.join(output_path, 'test_odds_ratio.parquet'))
#             del train_or, test_or, temp_df
#             gc.collect()
#             print("✓ Odds Ratio transformation complete and memory freed.\n")
#     else:
#         print("  - Skipping all supervised transformations: Target variable not provided.\n")

#     print("All transformations have been applied and saved to disk.")



In [88]:
# import pandas as pd
# import numpy as np
# from sklearn.preprocessing import PowerTransformer
# from typing import List, Dict, Tuple, Optional
# import gc

# def apply_transformations_memory_efficient(
#     train_df: pd.DataFrame, 
#     test_df: pd.DataFrame, 
#     target_series: pd.Series = None,
#     transformations: List[str] = None,
#     chunk_size: int = 1000,
#     save_to_disk: bool = False,
#     save_path: str = "./transformed_data/"
# ) -> Dict[str, Tuple[pd.DataFrame, pd.DataFrame]]:
#     """
#     Memory-efficient feature transformation that processes one transformation at a time.
    
#     Args:
#         train_df: Training dataframe with selected features
#         test_df: Testing dataframe with selected features  
#         target_series: Binary target series for supervised transformations
#         transformations: List of transformations to apply ['log', 'square', 'inverse', 'yeo-johnson', 'woe']
#         chunk_size: Number of columns to process at once for very wide datasets
#         save_to_disk: Whether to save transformed data to disk instead of keeping in memory
#         save_path: Path to save transformed datasets if save_to_disk=True
        
#     Returns:
#         Dictionary of transformation_name: (train_transformed, test_transformed)
#     """
    
#     if transformations is None:
#         transformations = ['log', 'square', 'inverse', 'yeo-johnson']
    
#     # Get numerical columns efficiently without copying data
#     numerical_cols = train_df.dtypes[train_df.dtypes.apply(lambda x: np.issubdtype(x, np.number))].index.tolist()
#     print(f"Found {len(numerical_cols)} numerical columns out of {len(train_df.columns)} total columns")
    
#     transformed_dfs = {}
#     epsilon = 1e-6
    
#     # Process each transformation separately to manage memory
#     for transform_name in transformations:
#         print(f"\n--- Applying {transform_name} transformation ---")
#         gc.collect()  # Clean up memory before each transformation
        
#         try:
#             if transform_name == 'log':
#                 train_transformed, test_transformed = apply_log_transform(
#                     train_df, test_df, numerical_cols, chunk_size
#                 )
                
#             elif transform_name == 'square':
#                 train_transformed, test_transformed = apply_square_transform(
#                     train_df, test_df, numerical_cols, chunk_size
#                 )
                
#             elif transform_name == 'inverse':
#                 train_transformed, test_transformed = apply_inverse_transform(
#                     train_df, test_df, numerical_cols, epsilon, chunk_size
#                 )
                
#             elif transform_name == 'yeo-johnson':
#                 train_transformed, test_transformed = apply_yeo_johnson_transform(
#                     train_df, test_df, numerical_cols, chunk_size
#                 )
                
#             elif transform_name == 'woe' and target_series is not None:
#                 train_transformed, test_transformed = apply_woe_transform(
#                     train_df, test_df, numerical_cols, target_series, chunk_size
#                 )
#             else:
#                 print(f"Skipping {transform_name}: Not implemented or missing target variable")
#                 continue
            
#             if save_to_disk:
#                 # Save to disk and store file paths instead of DataFrames
#                 import os
#                 os.makedirs(save_path, exist_ok=True)
#                 train_path = f"{save_path}train_{transform_name}.parquet"
#                 test_path = f"{save_path}test_{transform_name}.parquet"
#                 train_transformed.to_parquet(train_path)
#                 test_transformed.to_parquet(test_path)
#                 transformed_dfs[transform_name] = (train_path, test_path)
#                 print(f"✓ {transform_name} saved to disk")
#                 del train_transformed, test_transformed  # Free memory
#             else:
#                 transformed_dfs[transform_name] = (train_transformed, test_transformed)
#                 print(f"✓ {transform_name} transformation complete")
                
#         except MemoryError as e:
#             print(f"❌ Memory error during {transform_name} transformation: {e}")
#             print(f"Try reducing chunk_size or enabling save_to_disk=True")
#             continue
#         except Exception as e:
#             print(f"❌ Error during {transform_name} transformation: {e}")
#             continue
            
#         gc.collect()  # Clean up after each transformation
    
#     return transformed_dfs

# def apply_log_transform(train_df, test_df, numerical_cols, chunk_size):
#     """Apply log transformation in chunks"""
#     train_result = train_df.copy()
#     test_result = test_df.copy()
    
#     for i in range(0, len(numerical_cols), chunk_size):
#         chunk_cols = numerical_cols[i:i + chunk_size]
        
#         for col in chunk_cols:
#             if (train_result[col] >= 0).all():
#                 train_result[col] = np.log1p(train_result[col])
#                 test_result[col] = np.log1p(test_result[col])
#             else:
#                 print(f"  - Skipping log for '{col}' due to negative values")
                
#         if i % (chunk_size * 10) == 0:
#             print(f"  Processed {min(i + chunk_size, len(numerical_cols))}/{len(numerical_cols)} columns")
    
#     return train_result, test_result

# def apply_square_transform(train_df, test_df, numerical_cols, chunk_size):
#     """Apply square transformation in chunks"""
#     train_result = train_df.copy()
#     test_result = test_df.copy()
    
#     for i in range(0, len(numerical_cols), chunk_size):
#         chunk_cols = numerical_cols[i:i + chunk_size]
#         train_result[chunk_cols] = train_result[chunk_cols] ** 2
#         test_result[chunk_cols] = test_result[chunk_cols] ** 2
        
#         if i % (chunk_size * 10) == 0:
#             print(f"  Processed {min(i + chunk_size, len(numerical_cols))}/{len(numerical_cols)} columns")
    
#     return train_result, test_result

# def apply_inverse_transform(train_df, test_df, numerical_cols, epsilon, chunk_size):
#     """Apply inverse transformation in chunks"""
#     train_result = train_df.copy()
#     test_result = test_df.copy()
    
#     for i in range(0, len(numerical_cols), chunk_size):
#         chunk_cols = numerical_cols[i:i + chunk_size]
#         train_result[chunk_cols] = 1 / (train_result[chunk_cols] + epsilon)
#         test_result[chunk_cols] = 1 / (test_result[chunk_cols] + epsilon)
        
#         if i % (chunk_size * 10) == 0:
#             print(f"  Processed {min(i + chunk_size, len(numerical_cols))}/{len(numerical_cols)} columns")
    
#     return train_result, test_result

# def apply_yeo_johnson_transform(train_df, test_df, numerical_cols, chunk_size):
#     """Apply Yeo-Johnson transformation in chunks"""
#     train_result = train_df.copy()
#     test_result = test_df.copy()
    
#     # Process in chunks to manage memory
#     for i in range(0, len(numerical_cols), chunk_size):
#         chunk_cols = numerical_cols[i:i + chunk_size]
        
#         if len(chunk_cols) > 0:
#             pt = PowerTransformer(method='yeo-johnson', standardize=True)
#             train_chunk = pt.fit_transform(train_result[chunk_cols])
#             test_chunk = pt.transform(test_result[chunk_cols])
            
#             train_result[chunk_cols] = train_chunk
#             test_result[chunk_cols] = test_chunk
            
#         if i % (chunk_size * 5) == 0:
#             print(f"  Processed {min(i + chunk_size, len(numerical_cols))}/{len(numerical_cols)} columns")
    
#     return train_result, test_result

# def apply_woe_transform(train_df, test_df, numerical_cols, target_series, chunk_size):
#     """Apply WoE transformation (requires feature-engine)"""
#     try:
#         from feature_engine.encoding import WoEEncoder
        
#         train_result = train_df.copy()
#         test_result = test_df.copy()
        
#         # Process in smaller chunks for WoE
#         woe_chunk_size = min(chunk_size, 100)  # WoE is more memory intensive
        
#         for i in range(0, len(numerical_cols), woe_chunk_size):
#             chunk_cols = numerical_cols[i:i + woe_chunk_size]
            
#             woe_encoder = WoEEncoder(variables=chunk_cols)
#             train_chunk = woe_encoder.fit_transform(train_result[chunk_cols], target_series)
#             test_chunk = woe_encoder.transform(test_result[chunk_cols])
            
#             train_result[chunk_cols] = train_chunk[chunk_cols]
#             test_result[chunk_cols] = test_chunk[chunk_cols]
            
#             if i % (woe_chunk_size * 5) == 0:
#                 print(f"  Processed {min(i + woe_chunk_size, len(numerical_cols))}/{len(numerical_cols)} columns")
        
#         return train_result, test_result
        
#     except ImportError:
#         print("❌ feature-engine not installed. Install with: pip install feature-engine")
#         raise

# # Memory monitoring function
# def check_memory_usage():
#     """Check current memory usage"""
#     import psutil
#     process = psutil.Process()
#     memory_mb = process.memory_info().rss / 1024 / 1024
#     print(f"Current memory usage: {memory_mb:.1f} MB")
#     return memory_mb

# # Usage examples:
# def run_memory_efficient_transformations():
#     """Example usage with different memory management strategies"""
    
#     # Strategy 1: Process subset of transformations
#     print("=== Strategy 1: Subset of transformations ===")
#     transformations_subset = ['log', 'square']  # Start with fewer transformations
    
#     all_transformed_data = apply_transformations_memory_efficient(
#         train_final_df[features], 
#         test_final_df[features], 
#         target_series=train_final_df['target'],
#         transformations=transformations_subset,
#         chunk_size=500  # Smaller chunks for very wide data
#     )
    
#     # Strategy 2: Save to disk to manage memory
#     print("\n=== Strategy 2: Save to disk ===")
#     all_transformed_data_disk = apply_transformations_memory_efficient(
#         train_final_df[features], 
#         test_final_df[features], 
#         target_series=train_final_df['target'],
#         transformations=['yeo-johnson'],
#         chunk_size=200,
#         save_to_disk=True,
#         save_path="./transformed_features/"
#     )
    
#     # Strategy 3: Process one transformation at a time
#     print("\n=== Strategy 3: One at a time ===")
#     single_transformations = {}
    
#     for transform_name in ['log', 'square', 'inverse']:
#         print(f"\nProcessing {transform_name} only...")
#         check_memory_usage()
        
#         result = apply_transformations_memory_efficient(
#             train_final_df[features], 
#             test_final_df[features],
#             transformations=[transform_name],
#             chunk_size=1000
#         )
        
#         # Process or save the result immediately
#         if transform_name in result:
#             # Do something with the transformed data
#             train_transformed, test_transformed = result[transform_name]
#             print(f"✓ {transform_name} shape: {train_transformed.shape}")
            
#             # Save or use the data, then clean up
#             del train_transformed, test_transformed
            
#         gc.collect()
#         check_memory_usage()

# # Uncomment to run:
# # run_memory_efficient_transformations()

In [72]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import PowerTransformer
from typing import Tuple
import os
import gc

# You might need to install these libraries
# !pip install feature-engine
# !pip install pyarrow
try:
    from feature_engine.encoding import WoEEncoder
    from feature_engine.discretisation import EqualFrequencyDiscretiser
except ImportError:
    WoEEncoder = None
    EqualFrequencyDiscretiser = None


def apply_and_save_transformations(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    output_path: str,
    target_series: pd.Series = None
) -> None:
    """
    Applies a suite of transformations, saves each result to a Parquet file,
    and clears it from memory to prevent MemoryErrors.

    Args:
        train_df (pd.DataFrame): The training dataframe with selected features.
        test_df (pd.DataFrame): The testing dataframe with selected features.
        output_path (str): The directory path to save the output Parquet files.
        target_series (pd.Series): The binary target series required for
                                   supervised transformations.
    """
    # --- Setup ---
    # Create the output directory if it doesn't exist
    os.makedirs(output_path, exist_ok=True)
    
    numerical_cols = train_df.select_dtypes(include=np.number).columns.tolist()
    epsilon = 1e-6  # To prevent division by zero or log(0)

    # --- 1. Log Transform ---
    print("Applying Log transformation...")
    train_log = train_df.copy()
    test_log = test_df.copy()
    for col in numerical_cols:
        if (train_log[col] >= 0).all():
            train_log[col] = np.log1p(train_log[col])
            test_log[col] = np.log1p(test_log[col])
    print("  - Saving transformed data to Parquet...")
    train_log.to_parquet(os.path.join(output_path, 'train_log.parquet'))
    test_log.to_parquet(os.path.join(output_path, 'test_log.parquet'))
    del train_log, test_log
    gc.collect()
    print("✓ Log transformation complete and memory freed.\n")

    # --- 2. Square Transform ---
    print("Applying Square transformation...")
    train_sq = train_df.copy()
    test_sq = test_df.copy()
    train_sq[numerical_cols] = train_sq[numerical_cols]**2
    test_sq[numerical_cols] = test_sq[numerical_cols]**2
    print("  - Saving transformed data to Parquet...")
    train_sq.to_parquet(os.path.join(output_path, 'train_square.parquet'))
    test_sq.to_parquet(os.path.join(output_path, 'test_square.parquet'))
    del train_sq, test_sq
    gc.collect()
    print("✓ Square transformation complete and memory freed.\n")

    # --- 3. Inverse Transform ---
    print("Applying Inverse transformation...")
    train_inv = train_df.copy()
    test_inv = test_df.copy()
    train_inv[numerical_cols] = 1 / (train_inv[numerical_cols] + epsilon)
    test_inv[numerical_cols] = 1 / (test_inv[numerical_cols] + epsilon)
    print("  - Saving transformed data to Parquet...")
    train_inv.to_parquet(os.path.join(output_path, 'train_inverse.parquet'))
    test_inv.to_parquet(os.path.join(output_path, 'test_inverse.parquet'))
    del train_inv, test_inv
    gc.collect()
    print("✓ Inverse transformation complete and memory freed.\n")

    # --- 4. Log-Odds (Logit) Transform ---
    print("Applying Log-Odds transformation...")
    train_logodds = train_df.copy()
    test_logodds = test_df.copy()
    logodds_applied = False
    for col in numerical_cols:
        # This check is an approximation. A more robust check would be needed for large data.
        if (train_logodds[col] > 0).all() and (train_logodds[col] < 1).all():
            logodds_applied = True
            p_train = np.clip(train_logodds[col], epsilon, 1 - epsilon)
            p_test = np.clip(test_logodds[col], epsilon, 1 - epsilon)
            train_logodds[col] = np.log(p_train / (1 - p_train))
            test_logodds[col] = np.log(p_test / (1 - p_test))
    
    if logodds_applied:
        print("  - Saving transformed data to Parquet...")
        train_logodds.to_parquet(os.path.join(output_path, 'train_logodds.parquet'))
        test_logodds.to_parquet(os.path.join(output_path, 'test_logodds.parquet'))
        print("✓ Log-Odds transformation complete and memory freed.\n")
    else:
        print("  - Skipping Log-Odds: No columns with values in (0, 1) range found.\n")
    del train_logodds, test_logodds
    gc.collect()

    # --- 5. Yeo-Johnson Transform (Column by Column) ---
    print("Applying Yeo-Johnson transformation column by column...")
    if numerical_cols:
        train_yj = train_df.copy()
        test_yj = test_df.copy()
        
        for col in numerical_cols:
            try:
                pt = PowerTransformer(method='yeo-johnson', standardize=True)
                # Reshape column to 2D array for the transformer
                train_yj[col] = pt.fit_transform(train_yj[[col]])
                test_yj[col] = pt.transform(test_yj[[col]])
            except RuntimeError as e:
                # Catch RuntimeError and check the message to specifically handle BracketError
                if "finding a valid bracket" in str(e):
                    print(f"  - WARNING: Skipping Yeo-Johnson for column '{col}'. BracketError occurred (likely due to low variance).")
                else:
                    # Re-raise the error if it's a different RuntimeError
                    raise
            except Exception as e:
                print(f"  - WARNING: An unexpected error occurred for column '{col}': {e}")

        print("  - Saving transformed data to Parquet...")
        train_yj.to_parquet(os.path.join(output_path, 'train_yeo-johnson.parquet'))
        test_yj.to_parquet(os.path.join(output_path, 'test_yeo-johnson.parquet'))
        del train_yj, test_yj
        gc.collect()
        print("✓ Yeo-Johnson transformation complete and memory freed.\n")
    else:
        print("  - Skipping Yeo-Johnson: No numerical columns found.\n")

    # --- 6. Supervised Transformations ---
    if target_series is not None:
        # --- Weight of Evidence (WoE) Transformation (Column by Column) ---
        print("Applying Weight of Evidence (WoE) transformation...")
        if WoEEncoder is None or EqualFrequencyDiscretiser is None:
            print("  - Skipping WoE: `feature-engine` is not installed. Please run `pip install feature-engine`.")
        elif not numerical_cols:
            print("  - Skipping WoE: No numerical columns to apply on.")
        else:
            train_woe = train_df.copy()
            test_woe = test_df.copy()
            
            for col in numerical_cols:
                try:
                    # 1. Discretize the numerical variable into categories
                    discretizer = EqualFrequencyDiscretiser(q=10, variables=[col], return_object=True)
                    train_binned = discretizer.fit_transform(train_woe[[col]])
                    test_binned = discretizer.transform(test_woe[[col]])

                    # 2. Apply WoE to the now-binned (categorical) feature
                    woe_encoder = WoEEncoder(variables=[col])
                    train_woe[col] = woe_encoder.fit_transform(train_binned, target_series)[col]
                    test_woe[col] = woe_encoder.transform(test_binned)[col]

                except ValueError as e:
                    if "contained 0 in the denominator or numerator" in str(e):
                        print(f"  - WARNING: Skipping WoE for column '{col}'. A bin contained only one target class.")
                    else:
                        raise
                except Exception as e:
                    print(f"  - WARNING: An unexpected error occurred for column '{col}' during WoE: {e}")

            print("  - Saving transformed data to Parquet...")
            train_woe.to_parquet(os.path.join(output_path, 'train_woe.parquet'))
            test_woe.to_parquet(os.path.join(output_path, 'test_woe.parquet'))
            del train_woe, test_woe
            gc.collect()
            print("✓ WoE transformation complete and memory freed.\n")

        # --- Odds Ratio Transformation (Column by Column) ---
        print("Applying Odds Ratio transformation...")
        if EqualFrequencyDiscretiser is None:
             print("  - Skipping Odds Ratio: `feature-engine` is not installed.")
        elif not numerical_cols:
            print("  - Skipping Odds Ratio: No numerical columns found.\n")
        else:
            train_or = train_df.copy()
            test_or = test_df.copy()
            
            for col in numerical_cols:
                try:
                    # 1. Discretize the numerical variable
                    discretizer_or = EqualFrequencyDiscretiser(q=10, variables=[col], return_object=True)
                    train_or_binned = discretizer_or.fit_transform(train_or[[col]])
                    test_or_binned = discretizer_or.transform(test_or[[col]])

                    # 2. Calculate Odds Ratio on the binned data
                    temp_df = pd.concat([train_or_binned, target_series], axis=1)
                    target_name = target_series.name
                    
                    cross_tab = pd.crosstab(temp_df[col], temp_df[target_name])
                    if 0 not in cross_tab.columns: cross_tab[0] = 0
                    if 1 not in cross_tab.columns: cross_tab[1] = 0
                    
                    odds = (cross_tab[1] + epsilon) / (cross_tab[0] + epsilon)
                    
                    reference_category = train_or_binned[col].mode()[0]
                    reference_odds = odds.get(reference_category, epsilon)
                    
                    or_mapper = (odds / reference_odds).to_dict()
                    
                    # 3. Replace original values with the calculated Odds Ratio
                    train_or[col] = train_or_binned[col].map(or_mapper).fillna(1.0)
                    test_or[col] = test_or_binned[col].map(or_mapper).fillna(1.0)
                
                except Exception as e:
                     print(f"  - WARNING: An unexpected error occurred for column '{col}' during Odds Ratio: {e}")

            print("  - Saving transformed data to Parquet...")
            train_or.to_parquet(os.path.join(output_path, 'train_odds_ratio.parquet'))
            test_or.to_parquet(os.path.join(output_path, 'test_odds_ratio.parquet'))
            del train_or, test_or
            gc.collect()
            print("✓ Odds Ratio transformation complete and memory freed.\n")
    else:
        print("  - Skipping all supervised transformations: Target variable not provided.\n")

    print("All transformations have been applied and saved to disk.")


In [23]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import PowerTransformer
from typing import Tuple, Optional
import os
import gc

# You might need to install these libraries
# !pip install feature-engine
# !pip install pyarrow
try:
    from feature_engine.encoding import WoEEncoder
    from feature_engine.discretisation import EqualFrequencyDiscretiser
except ImportError:
    WoEEncoder = None
    EqualFrequencyDiscretiser = None


def apply_and_save_transformations(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    train_target: pd.Series,
    output_path: str,
    test_target: Optional[pd.Series] = None
) -> None:
    """
    Applies a suite of transformations, saves each result to a Parquet file
    (including targets), and clears it from memory to prevent MemoryErrors.

    Args:
        train_df (pd.DataFrame): The training dataframe with selected features.
        test_df (pd.DataFrame): The testing dataframe with selected features.
        train_target (pd.Series): The target series for training data.
        output_path (str): The directory path to save the output Parquet files.
        test_target (pd.Series, optional): The target series for test data.
                                          If None, test datasets will only contain features.
    """
    # --- Setup ---
    # Create the output directory if it doesn't exist
    os.makedirs(output_path, exist_ok=True)
    
    numerical_cols = train_df.select_dtypes(include=np.number).columns.tolist()
    epsilon = 1e-6  # To prevent division by zero or log(0)

    def save_with_target(train_transformed, test_transformed, filename_prefix):
        """Helper function to save datasets with targets included"""
        # Save training data with target
        train_with_target = pd.concat([train_transformed, train_target], axis=1)
        train_with_target.to_parquet(os.path.join(output_path, f'{filename_prefix}_train.parquet'))
        
        # Save test data (with target if available, otherwise just features)
        if test_target is not None:
            test_with_target = pd.concat([test_transformed, test_target], axis=1)
            test_with_target.to_parquet(os.path.join(output_path, f'{filename_prefix}_test.parquet'))
        else:
            test_transformed.to_parquet(os.path.join(output_path, f'{filename_prefix}_test.parquet'))

    # --- 1. Log Transform ---
    print("Applying Log transformation...")
    train_log = train_df.copy()
    test_log = test_df.copy()
    for col in numerical_cols:
        if (train_log[col] >= 0).all():
            train_log[col] = np.log1p(train_log[col])
            test_log[col] = np.log1p(test_log[col])
    
    print("  - Saving transformed data to Parquet...")
    save_with_target(train_log, test_log, 'log_transform')
    del train_log, test_log
    gc.collect()
    print("✓ Log transformation complete and memory freed.\n")

    # --- 2. Square Transform ---
    print("Applying Square transformation...")
    train_sq = train_df.copy()
    test_sq = test_df.copy()
    train_sq[numerical_cols] = train_sq[numerical_cols]**2
    test_sq[numerical_cols] = test_sq[numerical_cols]**2
    
    print("  - Saving transformed data to Parquet...")
    save_with_target(train_sq, test_sq, 'square_transform')
    del train_sq, test_sq
    gc.collect()
    print("✓ Square transformation complete and memory freed.\n")

    # --- 3. Inverse Transform ---
    print("Applying Inverse transformation...")
    train_inv = train_df.copy()
    test_inv = test_df.copy()
    train_inv[numerical_cols] = 1 / (train_inv[numerical_cols] + epsilon)
    test_inv[numerical_cols] = 1 / (test_inv[numerical_cols] + epsilon)
    
    print("  - Saving transformed data to Parquet...")
    save_with_target(train_inv, test_inv, 'inverse_transform')
    del train_inv, test_inv
    gc.collect()
    print("✓ Inverse transformation complete and memory freed.\n")

    # --- 4. Log-Odds (Logit) Transform ---
    print("Applying Log-Odds transformation...")
    train_logodds = train_df.copy()
    test_logodds = test_df.copy()
    logodds_applied = False
    for col in numerical_cols:
        # This check is an approximation. A more robust check would be needed for large data.
        if (train_logodds[col] > 0).all() and (train_logodds[col] < 1).all():
            logodds_applied = True
            p_train = np.clip(train_logodds[col], epsilon, 1 - epsilon)
            p_test = np.clip(test_logodds[col], epsilon, 1 - epsilon)
            train_logodds[col] = np.log(p_train / (1 - p_train))
            test_logodds[col] = np.log(p_test / (1 - p_test))
    
    if logodds_applied:
        print("  - Saving transformed data to Parquet...")
        save_with_target(train_logodds, test_logodds, 'logodds_transform')
        print("✓ Log-Odds transformation complete and memory freed.\n")
    else:
        print("  - Skipping Log-Odds: No columns with values in (0, 1) range found.\n")
    
    del train_logodds, test_logodds
    gc.collect()

    # --- 5. Yeo-Johnson Transform (Column by Column) ---
    print("Applying Yeo-Johnson transformation column by column...")
    if numerical_cols:
        train_yj = train_df.copy()
        test_yj = test_df.copy()
        
        for col in numerical_cols:
            try:
                pt = PowerTransformer(method='yeo-johnson', standardize=True)
                # Reshape column to 2D array for the transformer
                train_yj[col] = pt.fit_transform(train_yj[[col]])
                test_yj[col] = pt.transform(test_yj[[col]])
            except RuntimeError as e:
                # Catch RuntimeError and check the message to specifically handle BracketError
                if "finding a valid bracket" in str(e):
                    print(f"  - WARNING: Skipping Yeo-Johnson for column '{col}'. BracketError occurred (likely due to low variance).")
                else:
                    # Re-raise the error if it's a different RuntimeError
                    raise
            except Exception as e:
                print(f"  - WARNING: An unexpected error occurred for column '{col}': {e}")

        print("  - Saving transformed data to Parquet...")
        save_with_target(train_yj, test_yj, 'yeo_johnson_transform')
        del train_yj, test_yj
        gc.collect()
        print("✓ Yeo-Johnson transformation complete and memory freed.\n")
    else:
        print("  - Skipping Yeo-Johnson: No numerical columns found.\n")

    # --- 6. Supervised Transformations ---
    # --- Weight of Evidence (WoE) Transformation (Column by Column) ---
    print("Applying Weight of Evidence (WoE) transformation...")
    if WoEEncoder is None or EqualFrequencyDiscretiser is None:
        print("  - Skipping WoE: `feature-engine` is not installed. Please run `pip install feature-engine`.")
    elif not numerical_cols:
        print("  - Skipping WoE: No numerical columns to apply on.")
    else:
        train_woe = train_df.copy()
        test_woe = test_df.copy()
        
        for col in numerical_cols:
            try:
                # 1. Discretize the numerical variable into categories
                discretizer = EqualFrequencyDiscretiser(q=10, variables=[col], return_object=True)
                train_binned = discretizer.fit_transform(train_woe[[col]])
                test_binned = discretizer.transform(test_woe[[col]])

                # 2. Apply WoE to the now-binned (categorical) feature
                woe_encoder = WoEEncoder(variables=[col])
                train_woe[col] = woe_encoder.fit_transform(train_binned, train_target)[col]
                test_woe[col] = woe_encoder.transform(test_binned)[col]

            except ValueError as e:
                if "contained 0 in the denominator or numerator" in str(e):
                    print(f"  - WARNING: Skipping WoE for column '{col}'. A bin contained only one target class.")
                else:
                    raise
            except Exception as e:
                print(f"  - WARNING: An unexpected error occurred for column '{col}' during WoE: {e}")

        print("  - Saving transformed data to Parquet...")
        save_with_target(train_woe, test_woe, 'woe_transform')
        del train_woe, test_woe
        gc.collect()
        print("✓ WoE transformation complete and memory freed.\n")

    # --- Odds Ratio Transformation (Column by Column) ---
    print("Applying Odds Ratio transformation...")
    if EqualFrequencyDiscretiser is None:
         print("  - Skipping Odds Ratio: `feature-engine` is not installed.")
    elif not numerical_cols:
        print("  - Skipping Odds Ratio: No numerical columns found.\n")
    else:
        train_or = train_df.copy()
        test_or = test_df.copy()
        
        for col in numerical_cols:
            try:
                # 1. Discretize the numerical variable
                discretizer_or = EqualFrequencyDiscretiser(q=10, variables=[col], return_object=True)
                train_or_binned = discretizer_or.fit_transform(train_or[[col]])
                test_or_binned = discretizer_or.transform(test_or[[col]])

                # 2. Calculate Odds Ratio on the binned data
                temp_df = pd.concat([train_or_binned, train_target], axis=1)
                target_name = train_target.name
                
                cross_tab = pd.crosstab(temp_df[col], temp_df[target_name])
                if 0 not in cross_tab.columns: cross_tab[0] = 0
                if 1 not in cross_tab.columns: cross_tab[1] = 0
                
                odds = (cross_tab[1] + epsilon) / (cross_tab[0] + epsilon)
                
                reference_category = train_or_binned[col].mode()[0]
                reference_odds = odds.get(reference_category, epsilon)
                
                or_mapper = (odds / reference_odds).to_dict()
                
                # 3. Replace original values with the calculated Odds Ratio
                train_or[col] = train_or_binned[col].map(or_mapper).fillna(1.0)
                test_or[col] = test_or_binned[col].map(or_mapper).fillna(1.0)
            
            except Exception as e:
                 print(f"  - WARNING: An unexpected error occurred for column '{col}' during Odds Ratio: {e}")

        print("  - Saving transformed data to Parquet...")
        save_with_target(train_or, test_or, 'odds_ratio_transform')
        del train_or, test_or
        gc.collect()
        print("✓ Odds Ratio transformation complete and memory freed.\n")

    print("All transformations have been applied and saved to disk.")
    print(f"Files saved in: {output_path}")
    print("File naming convention: [transformation]_[train/test].parquet")


# Example usage:
"""
# Load your data
train_features = pd.read_csv('train_features.csv')
test_features = pd.read_csv('test_features.csv')
train_target = pd.read_csv('train.csv')['target']  # assuming target column is named 'target'

# Optional: load test target if available
test_target = pd.read_csv('test.csv')['target'] if 'target' in test_df.columns else None

# Apply transformations
apply_and_save_transformations(
    train_df=train_features,
    test_df=test_features,
    train_target=train_target,
    output_path='./transformed_datasets/',
    test_target=test_target  # Optional
)
"""

"\n# Load your data\ntrain_features = pd.read_csv('train_features.csv')\ntest_features = pd.read_csv('test_features.csv')\ntrain_target = pd.read_csv('train.csv')['target']  # assuming target column is named 'target'\n\n# Optional: load test target if available\ntest_target = pd.read_csv('test.csv')['target'] if 'target' in test_df.columns else None\n\n# Apply transformations\napply_and_save_transformations(\n    train_df=train_features,\n    test_df=test_features,\n    train_target=train_target,\n    output_path='./transformed_datasets/',\n    test_target=test_target  # Optional\n)\n"

In [14]:
%whos

Variable                         Type          Data/Info
--------------------------------------------------------
PowerTransformer                 type          <class 'sklearn.preproces<...>._data.PowerTransformer'>
Tuple                            _TupleType    typing.Tuple
apply_and_save_transformations   function      <function apply_and_save_<...>ations at 0x7f4826c06700>
duckdb                           module        <module 'duckdb' from '/g<...>ages/duckdb/__init__.py'>
features                         list          n=74
gc                               module        <module 'gc' (built-in)>
np                               module        <module 'numpy' from '/gp<...>kages/numpy/__init__.py'>
os                               module        <module 'os' (frozen)>
output_path                      str           /gpfs/user_home/os_home_dirs/zsherif2/test/
path                             str           /gpfs/user_home/os_home_d<...>rif2/test/transformations
pd                        

In [4]:
gc.collect()

0

In [24]:
features = [col for col in train_df.columns if col != 'target']

In [25]:
path = '/gpfs/user_home/os_home_dirs/zsherif2/test/transformations2'

In [26]:
apply_and_save_transformations(
    train_df=train_df[features],
    test_df=test_df[features],
    train_target=train_df['target'],
    output_path= path,
    test_target=test_df['target']  # Optional
)

Applying Log transformation...
  - Saving transformed data to Parquet...
✓ Log transformation complete and memory freed.

Applying Square transformation...
  - Saving transformed data to Parquet...
✓ Square transformation complete and memory freed.

Applying Inverse transformation...
  - Saving transformed data to Parquet...
✓ Inverse transformation complete and memory freed.

Applying Log-Odds transformation...
  - Skipping Log-Odds: No columns with values in (0, 1) range found.

Applying Yeo-Johnson transformation column by column...


/gpfs/user_home/os_home_dirs/zsherif2/.conda/envs/user_anaconda2024.02/lib/python3.12/site-packages/numpy/core/_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/gpfs/user_home/os_home_dirs/zsherif2/.conda/envs/user_anaconda2024.02/lib/python3.12/site-packages/numpy/core/_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/gpfs/user_home/os_home_dirs/zsherif2/.conda/envs/user_anaconda2024.02/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:3475: RuntimeWarning: overflow encountered in power
  out[pos] = (np.power(x[pos] + 1, lmbda) - 1) / lmbda


  - WARNING: Skipping Yeo-Johnson for column 'CRP02_0504128000'. BracketError occurred (likely due to low variance).


/gpfs/user_home/os_home_dirs/zsherif2/.conda/envs/user_anaconda2024.02/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:3475: RuntimeWarning: overflow encountered in power
  out[pos] = (np.power(x[pos] + 1, lmbda) - 1) / lmbda


  - WARNING: Skipping Yeo-Johnson for column 'CRP02_0402012001'. BracketError occurred (likely due to low variance).


...

  - WARNING: Skipping Yeo-Johnson for column 'CRP02_0404588001'. BracketError occurred (likely due to low variance).


/gpfs/user_home/os_home_dirs/zsherif2/.conda/envs/user_anaconda2024.02/lib/python3.12/site-packages/numpy/core/_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/gpfs/user_home/os_home_dirs/zsherif2/.conda/envs/user_anaconda2024.02/lib/python3.12/site-packages/numpy/core/_methods.py:187: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
/gpfs/user_home/os_home_dirs/zsherif2/.conda/envs/user_anaconda2024.02/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:3475: RuntimeWarning: overflow encountered in power
  out[pos] = (np.power(x[pos] + 1, lmbda) - 1) / lmbda


  - WARNING: Skipping Yeo-Johnson for column 'CRP02_0504472000'. BracketError occurred (likely due to low variance).


/gpfs/user_home/os_home_dirs/zsherif2/.conda/envs/user_anaconda2024.02/lib/python3.12/site-packages/numpy/core/_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/gpfs/user_home/os_home_dirs/zsherif2/.conda/envs/user_anaconda2024.02/lib/python3.12/site-packages/numpy/core/_methods.py:187: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
/gpfs/user_home/os_home_dirs/zsherif2/.conda/envs/user_anaconda2024.02/lib/python3.12/site-packages/numpy/core/_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)


  - Saving transformed data to Parquet...
✓ Yeo-Johnson transformation complete and memory freed.

Applying Weight of Evidence (WoE) transformation...
  - WARNING: Skipping WoE for column 'CRP02_0212535011'. A bin contained only one target class.
  - Saving transformed data to Parquet...
✓ WoE transformation complete and memory freed.

Applying Odds Ratio transformation...
  - Saving transformed data to Parquet...
✓ Odds Ratio transformation complete and memory freed.

All transformations have been applied and saved to disk.
Files saved in: /gpfs/user_home/os_home_dirs/zsherif2/test/transformations2
File naming convention: [transformation]_[train/test].parquet


In [38]:
# all_transformed_data = apply_and_save_transformations(
#     train_df[features]  , 
#     test_df[features]  , path, train_df['target'])

Applying Weight of Evidence (WoE) transformation...
  - WARNING: Skipping WoE for column 'CRP02_0214125000'. A bin contained only one target class.
  - WARNING: Skipping WoE for column 'CRP02_0214012000'. A bin contained only one target class.
  - Saving transformed data to Parquet...
✓ WoE transformation complete and memory freed.

Applying Odds Ratio transformation...
  - Saving transformed data to Parquet...
✓ Odds Ratio transformation complete and memory freed.

All transformations have been applied and saved to disk.


In [ ]:
# all_transformed_data = apply_transformations_memory_efficient(
#     final_train_df[features], 
#     final_train_df[features], final_train_df['target'])

In [ ]:
skew_results = {}

# First, get the skew of the original data for a baseline
skew_results['original'] = train_selected_df.skew()

for name, (train_df, test_df) in all_transformed_data.items():
    
   # Calculate skew for the current transformed training df and add it to the dictionary
    skew_results[name] = train_df.skew()
    
# Convert the dictionary of skew results into a single DataFrame
skew_df = pd.DataFrame(skew_results)
skew_df.head(20)

In [ ]:
# Set the figure size
plt.figure(figsize=(12, 5))

# Create the heatmap
sns.heatmap(
    skew_df, 
    annot=True,     # Display the skew values in the cells
    cmap='coolwarm',# Use a diverging colormap (blue=negative, red=positive, white=near zero)
    center=0,       # Center the colormap at zero
    fmt=".2f"       # Format annotations to two decimal places
)

# Add a title
plt.title('Heatmap of Skewness Across Transformations', fontsize=16)

# Display the plot
plt.show()